# Análisis Exploratorio de Datos — Anomalías en la Actividad Sísmica Regional

**Dataset:** `Significant_Earthquakes.xlsx` — catálogo USGS, ~115.500 sismos, 22 variables (1900–2023)

---

## 1. Planteamiento del problema

 ¿la actividad sísmica reciente de una región está
estadísticamente fuera de lo esperado** respecto a su comportamiento histórico?


---
# 2. ETL (Extracción, Transformación y Carga)

In [ ]:
import os
import warnings
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "font.size": 9,
})
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 170)

RANDOM_STATE = 42
ALPHA = 0.05
np.random.seed(RANDOM_STATE)


def titulo(t):
    print("\n" + "=" * 86)
    print(t.upper())
    print("=" * 86)


def conclusion(texto):
    print("\n" + "-" * 86)
    print("CONCLUSION ESTADISTICA")
    print("-" * 86)
    for parrafo in texto.strip().split("\n"):
        if parrafo.strip():
            print(textwrap.fill(parrafo.strip(), width=86))
    print("-" * 86)


def veredicto(p, alpha=ALPHA, h0="H0", h1="H1"):
    """Traduce un p-valor a una decisión estadística explícita."""
    if p < alpha:
        return f"p = {p:.3e} < {alpha}  ->  se RECHAZA {h0}. Conclusion: {h1}"
    return f"p = {p:.3e} >= {alpha}  ->  NO se rechaza {h0}"


print("Librerias cargadas correctamente.")
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2.1 Extracción 



In [ ]:

RUTA_DATOS = "Significant_Earthquakes.csv"

CANDIDATAS = [
    RUTA_DATOS,
    os.environ.get("RUTA_SISMOS"),
    "Significant_Earthquakes.xlsx",
    "data/Significant_Earthquakes.xlsx",
    "../data/Significant_Earthquakes.xlsx",
]

ruta = next((p for p in CANDIDATAS if p and Path(p).exists()), None)
if ruta is None:
    raise FileNotFoundError(
        "No se encontro Significant_Earthquakes.xlsx. "
        "Asigne la ruta completa a la constante RUTA_DATOS en esta celda."
    )

print("Leyendo:", ruta)
df_raw = pd.read_csv(ruta)
print("Dimensiones:", df_raw.shape)
df_raw.head()

In [ ]:
titulo("1. Auditoria tecnica del dataset crudo")

auditoria = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "no_nulos": df_raw.notna().sum(),
    "nulos": df_raw.isna().sum(),
    "%_nulos": (df_raw.isna().mean() * 100).round(2),
    "unicos": df_raw.nunique(),
})
display(auditoria.sort_values("%_nulos", ascending=False))

print("\nFilas duplicadas completas :", df_raw.duplicated().sum())
print("IDs duplicados             :", df_raw["id"].duplicated().sum())
print("Valores de 'type'          :", df_raw["type"].value_counts().head().to_dict())
print("Rango de magnitud          : [%.2f , %.2f]" % (df_raw["mag"].min(), df_raw["mag"].max()))

# Mapa visual de faltantes 
muestra = df_raw.sample(min(3000, len(df_raw)), random_state=RANDOM_STATE).sort_index()
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(muestra.isna(), cbar=False, cmap="viridis", yticklabels=False, ax=ax)
ax.set_title("Mapa de valores faltantes (amarillo = ausente) — muestra de 3.000 registros")
plt.tight_layout()
plt.show()

## 2.2 Transformación — limpieza y control de calidad del catálogo

In [ ]:
titulo("2. Limpieza y control de calidad")

df = df_raw.copy()
n0 = len(df)
reporte_limpieza = []


def paso(nombre, df_nuevo):
    """Aplica un filtro y registra cuantas filas cuesta."""
    global df
    reporte_limpieza.append({
        "paso": nombre,
        "filas": len(df_nuevo),
        "eliminadas": len(df) - len(df_nuevo),
    })
    df = df_nuevo
    return df_nuevo


df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True)
df["updated"] = pd.to_datetime(df["updated"], errors="coerce", utc=True)
for c in ["latitude", "longitude", "depth", "mag"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")


paso("dataset crudo", df)
paso("time valido", df[df["time"].notna()])
paso("solo type == 'earthquake'", df[df["type"].astype(str).str.lower() == "earthquake"])
paso("id unico", df.drop_duplicates(subset="id", keep="first"))
paso("duplicados espacio-temporales",
     df.drop_duplicates(subset=["time", "latitude", "longitude", "mag"], keep="first"))
paso("coordenadas validas",
     df[df["latitude"].between(-90, 90) & df["longitude"].between(-180, 180)])
paso("magnitud >= 5.0", df[df["mag"] >= 5.0])
paso("profundidad en [-5, 700] km o nula",
     df[df["depth"].isna() | df["depth"].between(-5, 700)])

rep = pd.DataFrame(reporte_limpieza)
rep["%_retenido"] = (rep["filas"] / n0 * 100).round(2)
display(rep)




### 2.2.3 Magnitud de completitud ($M_c$) y ventana temporal válida


In [ ]:
titulo("3. Completitud temporal del catalogo")

df["año"] = df["time"].dt.year
conteo_anual = df.groupby("año").size()

# Criterio objetivo: primer año a partir del cual el conteo se mantiene (>90% de los años)
# por encima del 50% del nivel instrumental moderno (mediana de los ultimos 30 años)
nivel_moderno = conteo_anual.loc[conteo_anual.index >= conteo_anual.index.max() - 30].median()
umbral = 0.5 * nivel_moderno
candidatos = [a for a in conteo_anual.index if (conteo_anual.loc[a:] >= umbral).mean() > 0.90]
AÑO_INICIO = int(min(candidatos)) if candidatos else 1973

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(conteo_anual.index, conteo_anual.values, width=0.9, color="#4C72B0")
ax.axhline(umbral, color="#C44E52", ls="--", lw=1.4,
           label=f"umbral de completitud ({umbral:.0f} eventos/año)")
ax.axvline(AÑO_INICIO, color="black", ls=":", lw=2,
           label=f"inicio de la ventana valida = {AÑO_INICIO}")
ax.set_title("Sismos M>=5.0 registrados por año — heterogeneidad instrumental del catalogo")
ax.set_xlabel("Año")
ax.set_ylabel("N. de sismos")
ax.legend()
plt.tight_layout()
plt.show()

antes = max(conteo_anual.loc[:AÑO_INICIO - 1].median(), 1)
despues = conteo_anual.loc[AÑO_INICIO:].median()
print(f"Mediana anual antes de {AÑO_INICIO}: {antes:,.0f}")
print(f"Mediana anual desde  {AÑO_INICIO}: {despues:,.0f}")




In [ ]:
AÑO_FIN = int(df["año"].max())
print("Ultimo evento del catalogo:", df["time"].max())

dfq = df[df["año"] >= AÑO_INICIO].copy()   # quality catalog (catalogo de trabajo)

print(f"\nCatalogo de trabajo: {len(dfq):,} sismos  |  {AÑO_INICIO} - {AÑO_FIN}")
print(f"Descartados por completitud: {len(df) - len(dfq):,} sismos anteriores a {AÑO_INICIO}")

## 2.3 Ingeniería de variables

Se derivan las variables que la pregunta de investigación necesita y que **no existen** en el
archivo original:

| Variable | Definición | Justificación |
|---|---|---|
| `mes` | periodo mensual del evento | unidad temporal del panel |
| `log10_E` | $1.5\,M + 4.8$ (Hanks & Kanamori, 1979) | la magnitud es **logarítmica**; la energía es la cantidad física **aditiva** |
| `region` | celda de malla de 5°×5° | partición **objetiva y exhaustiva** del espacio |
| `region_nombre` | topónimo modal de `place` en la celda | interpretabilidad |
| `rango_prof` | superficial (<70 km), intermedia (70–300), profunda (>300) | clasificación sismológica estándar |
| `dt_horas` | tiempo transcurrido desde el sismo anterior en la región | diagnóstico de agrupamiento (réplicas) |

In [ ]:
titulo("4. Ingenieria de variables")

dfq["mes"] = dfq["time"].dt.to_period("M")
dfq["anio"] = dfq["time"].dt.year

dfq["log10_E"] = 1.5 * dfq["mag"] + 4.8

PASO_MALLA = 5
dfq["lat_bin"] = (np.floor(dfq["latitude"] / PASO_MALLA) * PASO_MALLA).astype(int)
dfq["lon_bin"] = (np.floor(dfq["longitude"] / PASO_MALLA) * PASO_MALLA).astype(int)
dfq["region"] = dfq["lat_bin"].astype(str) + "|" + dfq["lon_bin"].astype(str)


def toponimo(s):
    if not isinstance(s, str):
        return np.nan
    return s.split(",")[-1].strip() if "," in s else s.split(" of ")[-1].strip()


dfq["toponimo"] = dfq["place"].map(toponimo)
nombres = (dfq.dropna(subset=["toponimo"])
              .groupby("region")["toponimo"]
              .agg(lambda s: s.value_counts().index[0]))
dfq["region_nombre"] = dfq["region"].map(nombres).fillna("sin nombre")

CORTES_PROF = [-10, 70, 300, 1000]
ETIQ_PROF = ["Superficial (<70 km)", "Intermedia (70-300 km)", "Profunda (>300 km)"]
dfq["rango_prof"] = pd.cut(dfq["depth"], bins=CORTES_PROF, labels=ETIQ_PROF)

dfq = dfq.sort_values(["region", "time"])
dfq["dt_horas"] = (dfq.groupby("region")["time"].diff().dt.total_seconds() / 3600.0)
dfq = dfq.sort_values("time").reset_index(drop=True)

print(f"Regiones (celdas 5x5 grados) con al menos un sismo: {dfq['region'].nunique()}")
display(dfq[["time", "mes", "mag", "log10_E", "depth", "rango_prof",
             "region", "region_nombre", "dt_horas"]].head())

## 2.4 Gestión de datos faltantes


| Variable | % faltante | Mecanismo | Decisión |
|---|---|---|---|
| `nst`, `gap`, `dmin`, `horizontalError`, `magError`, `magNst`, `depthError`, `rms` | 25–71 % | **MNAR** (no reportado por redes antiguas) | **Eliminar** — imputar sesgaría |
| `depth` | 0,25 % | MAR | **Imputar** |
| `place` | 0,77 % | MCAR | **Etiquetar** como "desconocido" |



In [ ]:
titulo("5. Comparacion empirica de tecnicas de imputacion sobre 'depth'")

mask_faltante = dfq["depth"].isna()
print(f"Faltantes reales en depth: {mask_faltante.sum():,} ({mask_faltante.mean() * 100:.3f} %)")

observados = dfq.loc[~mask_faltante, "depth"]
idx_test = observados.sample(frac=0.05, random_state=RANDOM_STATE).index
verdad = dfq.loc[idx_test, "depth"].to_numpy()

if "region" not in dfq.columns:
    if not {"latitude", "longitude"}.issubset(dfq.columns):
        raise KeyError(
            "No se puede crear 'region': faltan 'latitude' y/o 'longitude'."
        )

    dfq["lat_bin"] = (
        np.floor(dfq["latitude"] / PASO_MALLA) * PASO_MALLA
    ).astype("Int64")
    dfq["lon_bin"] = (
        np.floor(dfq["longitude"] / PASO_MALLA) * PASO_MALLA
    ).astype("Int64")
    dfq["region"] = (
        dfq["lat_bin"].astype(str) + "|" + dfq["lon_bin"].astype(str)
    )

ensayo = dfq[["depth", "region", "time"]].copy()
ensayo.loc[idx_test, "depth"] = np.nan

imputaciones = {
    "Media global": ensayo["depth"].fillna(ensayo["depth"].mean()),
    "Mediana global": ensayo["depth"].fillna(ensayo["depth"].median()),
    "Mediana por region": ensayo["depth"].fillna(
        ensayo.groupby("region")["depth"].transform("median")),
    "Backward fill (bfill)": ensayo["depth"].bfill(),
    "Interpolacion cuadratica": ensayo["depth"].interpolate(method="quadratic",
                                                            limit_direction="both"),
}

filas = []
for nombre, serie in imputaciones.items():
    pred = serie.loc[idx_test].fillna(ensayo["depth"].median()).to_numpy()
    filas.append({
        "Tecnica": nombre,
        "MAE": np.mean(np.abs(pred - verdad)),
        "RMSE": np.sqrt(np.mean((pred - verdad) ** 2)),
        "Sesgo medio": np.mean(pred - verdad),
        "Cambio en desv.est. (%)": (np.std(pred) / np.std(verdad) - 1) * 100,
    })

comparacion = pd.DataFrame(filas).sort_values("RMSE").reset_index(drop=True).round(3)

comparacion["preserva_dispersion"] = comparacion["Cambio en desv.est. (%)"].abs() <= 25
admisibles = comparacion[comparacion["preserva_dispersion"]]
TECNICA_ELEGIDA = (admisibles.iloc[0]["Tecnica"] if len(admisibles)
                   else "Mediana por region")
display(comparacion)

ganadora_rmse = comparacion.iloc[0]["Tecnica"]
delta_media = comparacion.set_index("Tecnica")["Cambio en desv.est. (%)"]["Media global"]



In [ ]:
COLS_DESCARTE = ["nst", "gap", "dmin", "rms", "horizontalError",
                 "depthError", "magError", "magNst"]

if TECNICA_ELEGIDA == "Media global":
    dfq["depth"] = dfq["depth"].fillna(dfq["depth"].mean())
elif TECNICA_ELEGIDA == "Mediana global":
    dfq["depth"] = dfq["depth"].fillna(dfq["depth"].median())
elif TECNICA_ELEGIDA == "Backward fill (bfill)":
    dfq["depth"] = dfq["depth"].bfill()
elif TECNICA_ELEGIDA == "Interpolacion cuadratica":
    dfq["depth"] = dfq["depth"].interpolate(method="quadratic", limit_direction="both")
else:   # Mediana por region (opcion por defecto)
    dfq["depth"] = dfq["depth"].fillna(dfq.groupby("region")["depth"].transform("median"))

dfq["depth"] = dfq["depth"].fillna(dfq["depth"].median())   # red de seguridad
dfq["place"] = dfq["place"].fillna("desconocido")
dfq["rango_prof"] = pd.cut(dfq["depth"], bins=CORTES_PROF, labels=ETIQ_PROF)

dfm = dfq.drop(columns=[c for c in COLS_DESCARTE if c in dfq.columns])   # dataset de modelado

print(f"Tecnica de imputacion aplicada a 'depth': {TECNICA_ELEGIDA}")
VARS_MODELADO = ["time", "latitude", "longitude", "depth", "mag", "log10_E",
                 "region", "mes", "rango_prof"]
pendientes = dfm[VARS_MODELADO].isna().sum()[lambda s: s > 0]
print("\nFaltantes remanentes en las VARIABLES DE MODELADO:")
print(pendientes if len(pendientes) else "  ninguno — el dataset de modelado esta completo")
print("\nNota: 'dt_horas' conserva un NaN por region (el primer sismo de cada region no tiene "
      "evento previo). Es un faltante ESTRUCTURAL, no un dato perdido: no se imputa.")
print("\nDimensiones del dataset de modelado:", dfm.shape)

## 2.5 Construcción del panel región × mes

El catálogo es una **lista de eventos**; el modelo de Poisson necesita **conteos por unidad de
tiempo y espacio**. Se construye la rejilla **completa**, incluyendo explícitamente **los meses
con cero sismos**: omitirlos sesgaría la tasa histórica hacia arriba y haría que casi ninguna
región pareciera anómala.

In [ ]:
titulo("6. Panel region x mes")

# Solo regiones con actividad suficiente para hacer inferencia
MIN_EVENTOS_REGION = 60
regiones_validas = dfm["region"].value_counts()[lambda s: s >= MIN_EVENTOS_REGION].index
panel_base = dfm[dfm["region"].isin(regiones_validas)].copy()

print(f"Regiones retenidas: {len(regiones_validas)} de {dfm['region'].nunique()} "
      f"(criterio: >= {MIN_EVENTOS_REGION} sismos)")
print(f"Cobertura: {len(panel_base) / len(dfm) * 100:.1f} % de los eventos del catalogo")

# Rejilla completa region x mes (incluye los ceros)
meses = pd.period_range(panel_base["mes"].min(), panel_base["mes"].max(), freq="M")
malla = pd.MultiIndex.from_product([regiones_validas, meses], names=["region", "mes"])

agregado = panel_base.groupby(["region", "mes"]).agg(
    n_eventos=("mag", "size"),
    mag_max=("mag", "max"),
    mag_media=("mag", "mean"),
    prof_media=("depth", "mean"),
    log10_E_total=("log10_E", lambda s: float(np.log10(np.sum(10.0 ** s.astype(float))))),
)

panel = agregado.reindex(malla)
panel["n_eventos"] = panel["n_eventos"].fillna(0).astype(int)
panel = panel.reset_index()
panel["fecha"] = panel["mes"].dt.to_timestamp()
panel["region_nombre"] = panel["region"].map(nombres).fillna("sin nombre")
panel["region_label"] = panel["region_nombre"] + " [" + panel["region"] + "]"

print(f"\nPanel: {panel.shape[0]:,} filas = {len(regiones_validas)} regiones x {len(meses)} meses")
print(f"Meses con cero sismos: {(panel['n_eventos'] == 0).mean() * 100:.1f} % del panel")
display(panel.head())

## 2.6 División de los datos: 80 % histórico / 20 % reciente

> **Decisión metodológica crítica: la división NO puede ser aleatoria.**
>
> Una partición aleatoria mezclaría meses de 1990 con meses de 2023 en ambos conjuntos,
> destruyendo el orden temporal y produciendo *data leakage*: el "histórico" contendría
> información del futuro. Como la pregunta es explícitamente **"reciente vs. histórico"**,
> la división debe ser **cronológica**:
>
> - **80 % más antiguo → conjunto HISTÓRICO (*train*)**: línea base de la que se estiman
>   $\lambda_r$, el *b-value* y los límites de control.
> - **20 % más reciente → conjunto RECIENTE (*test*)**: periodo bajo evaluación, **nunca**
>   usado para estimar la línea base.
>
> Esta partición cumple el lineamiento 80/20 **y** define operativamente la pregunta de
> investigación.

In [ ]:
titulo("7. Division temporal 80 / 20")

corte_idx = int(len(meses) * 0.80)
MES_CORTE = meses[corte_idx]
print("Mes de corte:", MES_CORTE)

panel["periodo"] = np.where(panel["mes"] < MES_CORTE, "HISTORICO", "RECIENTE")
dfm["periodo"] = np.where(dfm["mes"] < MES_CORTE, "HISTORICO", "RECIENTE")

panel_hist = panel[panel["periodo"] == "HISTORICO"].copy()
panel_rec = panel[panel["periodo"] == "RECIENTE"].copy()
eventos_hist = dfm[dfm["periodo"] == "HISTORICO"]
eventos_rec = dfm[dfm["periodo"] == "RECIENTE"]

M_HIST = panel_hist["mes"].nunique()
M_REC = panel_rec["mes"].nunique()

resumen_split = pd.DataFrame({
    "Conjunto": ["HISTORICO (train)", "RECIENTE (test)"],
    "Desde": [str(meses[0]), str(MES_CORTE)],
    "Hasta": [str(meses[corte_idx - 1]), str(meses[-1])],
    "Meses": [M_HIST, M_REC],
    "% meses": [round(M_HIST / len(meses) * 100, 1), round(M_REC / len(meses) * 100, 1)],
    "Sismos": [len(eventos_hist), len(eventos_rec)],
    "Filas del panel": [len(panel_hist), len(panel_rec)],
})
display(resumen_split)

conclusion(
    f"El conjunto HISTORICO cubre {M_HIST} meses ({M_HIST / 12:.1f} anios) y el RECIENTE "
    f"{M_REC} meses ({M_REC / 12:.1f} anios). Toda estimacion de linea base (tasa lambda, "
    f"b-value, limites de control) se hara EXCLUSIVAMENTE con el historico; el conjunto reciente "
    f"es out-of-sample puro.\n"
    f"Esto evita la circularidad logica de declarar 'anomalo' un periodo que ya habia contribuido "
    f"a definir lo que es normal."
)

---
# 3. Fase 2 — Análisis univariado (caracterización)

Se caracteriza cada variable **por separado**: tendencia central, dispersión, **forma**
(asimetría y curtosis) y visualización (histograma normalizado + KDE + boxplot).

Todo el análisis univariado se hace **sobre el conjunto HISTÓRICO**, que es el que define
"lo normal". El conjunto reciente se reserva para la fase de contraste.

In [ ]:
titulo("8. Estadisticos descriptivos completos (conjunto HISTORICO)")

VARS_NUM = ["mag", "depth", "log10_E", "latitude", "longitude"]
base_uni = eventos_hist


def descriptivos(serie, nombre):
    s = pd.Series(serie).dropna().astype(float)
    q1, q3 = s.quantile([0.25, 0.75])
    return {
        "Variable": nombre,
        "n": len(s),
        "Media": s.mean(),
        "Mediana": s.median(),
        "Moda": s.mode().iloc[0] if len(s.mode()) else np.nan,
        "Desv.Est.": s.std(),
        "CV (%)": s.std() / s.mean() * 100 if s.mean() != 0 else np.nan,
        "Min": s.min(),
        "P05": s.quantile(0.05),
        "P25": q1,
        "P75": q3,
        "P95": s.quantile(0.95),
        "Max": s.max(),
        "IQR": q3 - q1,
        "Asimetria": stats.skew(s),
        "Curtosis (exceso)": stats.kurtosis(s),   # Fisher: 0 = normal
    }


tabla_uni = pd.DataFrame([descriptivos(base_uni[v], v) for v in VARS_NUM]
                         + [descriptivos(panel_hist["n_eventos"], "n_eventos (TARGET)")]
                         + [descriptivos(dfm.loc[dfm["periodo"] == "HISTORICO", "dt_horas"],
                                         "dt_horas")])
display(tabla_uni.set_index("Variable").T.round(4))

In [ ]:
titulo("9. Analisis de FORMA: asimetria y curtosis")


def leer_asimetria(a):
    if abs(a) < 0.5:
        return "aproximadamente simetrica"
    if abs(a) < 1:
        return "asimetria moderada a la " + ("DERECHA (cola larga superior)" if a > 0
                                             else "IZQUIERDA (cola larga inferior)")
    return "asimetria FUERTE a la " + ("DERECHA (cola larga superior)" if a > 0
                                       else "IZQUIERDA (cola larga inferior)")


def leer_curtosis(k):
    if k > 0.5:
        return f"LEPTOCURTICA (k={k:+.2f}): colas pesadas, exceso de valores extremos"
    if k < -0.5:
        return f"PLATICURTICA (k={k:+.2f}): colas ligeras, distribucion aplanada"
    return f"MESOCURTICA (k={k:+.2f}): colas similares a la normal"


for _, fila in tabla_uni.iterrows():
    print(f"\n{fila['Variable']}")
    print(f"   Asimetria = {fila['Asimetria']:+.3f}  ->  {leer_asimetria(fila['Asimetria'])}")
    print(f"   Curtosis  = {fila['Curtosis (exceso)']:+.3f}  ->  "
          f"{leer_curtosis(fila['Curtosis (exceso)'])}")
    # Coeficiente de asimetria de Pearson: 3*(media - mediana)/desv.est.
    rel = 3 * (fila["Media"] - fila["Mediana"]) / fila["Desv.Est."]
    print(f"   Asimetria de Pearson = {rel:+.3f}  ->  "
          + ("media arrastrada por la cola derecha; usar MEDIANA como medida de centro"
             if rel > 0.15 else
             "media arrastrada por la cola izquierda; usar MEDIANA como medida de centro"
             if rel < -0.15 else
             "centro estable: media y mediana coinciden"))

k_mag = float(tabla_uni.set_index("Variable").loc["mag", "Curtosis (exceso)"])
a_mag = float(tabla_uni.set_index("Variable").loc["mag", "Asimetria"])
conclusion(
    f"La magnitud presenta asimetria positiva ({a_mag:+.2f}) y curtosis {k_mag:+.2f}: NO es "
    f"gaussiana, es una distribucion de tipo EXPONENCIAL truncada en Mc = 5.0. Este resultado no "
    f"es una anomalia del dato: es exactamente lo que predice la LEY DE GUTENBERG-RICHTER "
    f"(seccion 3.3), que establece que la frecuencia de sismos decae exponencialmente con la "
    f"magnitud.\n"
    f"Consecuencia metodologica: cualquier tecnica que asuma normalidad de la magnitud (media +- "
    f"desviacion estandar, intervalos t de Student, regresion lineal ordinaria) esta mal "
    f"especificada. Se deben usar estimadores de maxima verosimilitud exponencial y estadistica "
    f"no parametrica."
)

## 3.1 Visualización univariada: histograma normalizado + KDE + boxplot

In [ ]:
def panel_univariado(serie, nombre, unidad="", bins=50, log_x=False):
    """Histograma normalizado + densidad de kernel + boxplot para una variable."""
    s = pd.Series(serie).dropna().astype(float)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(10, 5.4), sharex=True,
        gridspec_kw={"height_ratios": [3.2, 1], "hspace": 0.08})

    sns.histplot(s, bins=bins, stat="density", kde=True, color="#4C72B0",
                 edgecolor="white", linewidth=0.3, ax=ax1,
                 line_kws={"lw": 2.2, "color": "#C44E52"})
    ax1.axvline(s.mean(), color="#55A868", ls="--", lw=1.8, label=f"Media = {s.mean():.2f}")
    ax1.axvline(s.median(), color="#8172B2", ls="-.", lw=1.8, label=f"Mediana = {s.median():.2f}")
    ax1.set_ylabel("Densidad")
    ax1.set_title(f"{nombre} — histograma normalizado + KDE  "
                  f"(asimetria={stats.skew(s):+.2f}, curtosis={stats.kurtosis(s):+.2f})")
    ax1.legend(fontsize=8)

    sns.boxplot(x=s, ax=ax2, color="#4C72B0", width=0.5,
                flierprops={"markersize": 2.5, "alpha": 0.35})
    ax2.set_xlabel(f"{nombre} {unidad}".strip())
    if log_x:
        ax1.set_xscale("log")
        ax2.set_xscale("log")

    # Regla de Tukey para atipicos
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < lo) | (s > hi)).sum())
    plt.tight_layout()
    plt.show()
    print(f"   Bigotes de Tukey: [{lo:.2f} , {hi:.2f}]  |  "
          f"atipicos: {n_out:,} ({n_out / len(s) * 100:.2f} %)")
    return n_out / len(s) * 100


titulo("10. Distribucion de la MAGNITUD")
pct_out_mag = panel_univariado(base_uni["mag"], "Magnitud", "(Mw)", bins=45)
conclusion(
    f"La magnitud decae monotonamente desde Mc = 5.0: hay ordenes de magnitud mas sismos "
    f"pequenos que grandes. El boxplot marca {pct_out_mag:.2f} % de observaciones como 'atipicas' "
    f"por la regla de Tukey, PERO ESE DIAGNOSTICO ES INCORRECTO EN ESTE CONTEXTO: la regla de "
    f"1.5*IQR supone simetria, y aqui la cola derecha es intrinseca al fenomeno fisico. Un sismo "
    f"de M 8.5 no es un error de medicion ni un dato a eliminar: es precisamente el evento que mas "
    f"importa detectar.\n"
    f"DECISION: NO se eliminan magnitudes altas. Los valores extremos se tratan como senal, y la "
    f"deteccion de anomalias se hara sobre la TASA DE OCURRENCIA (conteos), no descartando eventos."
)

In [ ]:
titulo("11. Distribucion de la PROFUNDIDAD")
pct_out_prof = panel_univariado(base_uni["depth"], "Profundidad", "(km)", bins=70)

modas = base_uni["depth"].value_counts().head(5)
print("\n   Valores mas repetidos de profundidad:")
for v, c in modas.items():
    print(f"      {v:>7.1f} km : {c:>7,} sismos ({c / len(base_uni) * 100:5.2f} %)")

print("\n   Reparto por rango sismologico:")
display(base_uni["rango_prof"].value_counts(normalize=True).mul(100).round(2).to_frame("%"))

conclusion(
    f"La profundidad es fuertemente asimetrica a la derecha y MULTIMODAL. Los picos en 10 km y "
    f"33 km no son un hecho geofisico sino ARTEFACTOS ADMINISTRATIVOS: son los valores por "
    f"defecto que fija el USGS cuando la profundidad no puede resolverse con las fases "
    f"disponibles. Esto confirma que la media no es un resumen valido para esta variable y "
    f"justifica la imputacion por mediana adoptada en el ETL.\n"
    f"La estructura trimodal (superficial / intermedia / profunda) refleja la geometria de las "
    f"zonas de subduccion (planos de Wadati-Benioff), por lo que 'rango_prof' es una variable "
    f"categorica con contenido fisico real y no una discretizacion arbitraria."
)

In [ ]:
titulo("12. Distribucion del TARGET: conteo mensual de sismos por region")

target_hist = panel_hist["n_eventos"]
pct_out_n = panel_univariado(target_hist, "N.o de sismos por region-mes", bins=40)

media_t, var_t = target_hist.mean(), target_hist.var()
print(f"\n   Media    = {media_t:.4f}")
print(f"   Varianza = {var_t:.4f}")
print(f"   Razon varianza/media = {var_t / media_t:.3f}   (Poisson puro => 1.0)")
print(f"   Proporcion de ceros observada = {(target_hist == 0).mean():.4f}")
print(f"   Proporcion de ceros esperada bajo Poisson = {np.exp(-media_t):.4f}")

conclusion(
    f"El target es una variable de CONTEO: discreta, no negativa, con un {(target_hist == 0).mean() * 100:.1f} % "
    f"de ceros y una cola derecha muy larga. Es imposible que sea normal, por lo que el modelo de "
    f"referencia no es la gaussiana sino la POISSON.\n"
    f"Primer diagnostico clave: la razon varianza/media vale {var_t / media_t:.2f}. Si fuera 1, la "
    f"Poisson simple seria adecuada. "
    + (f"Como es MAYOR que 1, hay SOBREDISPERSION: los sismos no ocurren de forma independiente, "
       f"se agrupan en secuencias de replicas (leyes de Omori y de Bath). Ignorar esto haria que "
       f"el test de Poisson declarara anomalas a demasiadas regiones (inflacion del error tipo I). "
       f"En la seccion 6 se corrige mediante un ajuste CUASI-POISSON."
       if var_t / media_t > 1.2 else
       f"Como es cercana a 1, la aproximacion de Poisson es razonable.")
)

In [ ]:
titulo("13. Distribucion de los TIEMPOS ENTRE SISMOS CONSECUTIVOS (por region)")

dt = dfm.loc[(dfm["periodo"] == "HISTORICO") & dfm["dt_horas"].notna()
             & (dfm["dt_horas"] > 0), "dt_horas"]
pct_out_dt = panel_univariado(dt, "Tiempo entre sismos", "(horas, escala log)",
                              bins=60, log_x=True)
conclusion(
    f"Los tiempos entre eventos abarcan varios ordenes de magnitud (de minutos a anios), de ahi "
    f"la escala logaritmica. Bajo un proceso de Poisson HOMOGENEO, estos tiempos deberian seguir "
    f"una distribucion EXPONENCIAL con tasa constante. El exceso de tiempos muy cortos que se "
    f"observa a la izquierda es la firma estadistica del AGRUPAMIENTO POR REPLICAS. Este supuesto "
    f"se contrasta formalmente con un test de Kolmogorov-Smirnov en la seccion 5.3."
)

## 3.2 Variables categóricas

In [ ]:
titulo("14. Frecuencias de las variables categoricas")

VARS_CAT = ["magType", "rango_prof", "status", "region_nombre"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, v in zip(axes.ravel(), VARS_CAT):
    conteo = base_uni[v].value_counts().head(12)
    ax.barh(
        conteo.index.astype(str),
        conteo.values,
        color=sns.color_palette("viridis", n_colors=len(conteo))
    )
    ax.set_title(f"{v} — top {len(conteo)} categorias")
    ax.set_xlabel("N.o de sismos")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

for v in VARS_CAT:
    tabla = base_uni[v].value_counts(normalize=True).mul(100).round(2)
    print(f"\n{v}: {base_uni[v].nunique()} categorias | "
          f"top-3 = {tabla.head(3).to_dict()}")

conclusion(
    f"'magType' revela que el catalogo MEZCLA ESCALAS DE MAGNITUD distintas (mb, mw, mww, ms...), "
    f"que no son numericamente equivalentes entre si: mb satura por encima de M~6.5 mientras que "
    f"mww no. Esto es una fuente de heterogeneidad que se debe verificar (seccion 4.3) antes de "
    f"comparar magnitudes de epocas diferentes.\n"
    f"La concentracion geografica es extrema: unas pocas regiones (Cinturon de Fuego del "
    f"Pacifico) aportan la mayor parte de los eventos. Esto justifica que el analisis de "
    f"anomalias se haga POR REGION y no de forma global: una tasa mundial promedio no describe a "
    f"ninguna region en particular."
)

## 3.3 Ley de Gutenberg–Richter: el modelo clásico de la distribución de magnitudes



In [ ]:
titulo("15. Ley de Gutenberg-Richter y estimacion del b-value")

MC = 5.0            # magnitud de completitud del catalogo
DELTA_M = 0.1       # resolucion de binning de la magnitud


def b_value_aki(magnitudes, mc=MC, dm=DELTA_M):
    """Estimador MLE de Aki (1965) del b-value, con su error estandar."""
    m = np.asarray(pd.Series(magnitudes).dropna(), dtype=float)
    m = m[m >= mc]
    n = len(m)
    if n < 30:
        return np.nan, np.nan, n
    media = m.mean()
    denom = media - (mc - dm / 2.0)
    if denom <= 0:
        return np.nan, np.nan, n
    b = np.log10(np.e) / denom
    sigma = b / np.sqrt(n)                      # error estandar de Aki
    return b, sigma, n


b_glob, sb_glob, n_glob = b_value_aki(base_uni["mag"])
a_glob = np.log10(n_glob) + b_glob * MC

# --- Grafico: frecuencia acumulada y no acumulada ---------------------------
bins = np.arange(MC, base_uni["mag"].max() + DELTA_M, DELTA_M)
h, bordes = np.histogram(base_uni["mag"], bins=bins)
centros = bordes[:-1]
acum = h[::-1].cumsum()[::-1]

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.scatter(centros, acum, s=26, color="#4C72B0", label="Frecuencia acumulada  N(>=M)")
ax.scatter(centros, np.where(h > 0, h, np.nan), s=14, color="#CCB974", alpha=0.65,
           label="Frecuencia no acumulada")
mm = np.linspace(MC, base_uni["mag"].max(), 100)
ax.plot(mm, 10 ** (a_glob - b_glob * mm), color="#C44E52", lw=2.4,
        label=f"Ajuste G-R:  log10 N = {a_glob:.2f} - {b_glob:.3f} M")
ax.set_yscale("log")
ax.set_xlabel("Magnitud M")
ax.set_ylabel("N.o de sismos (escala log)")
ax.set_title("Ley de Gutenberg-Richter — conjunto HISTORICO")
ax.legend()
plt.tight_layout()
plt.show()

print(f"b-value global (MLE de Aki) = {b_glob:.4f} +- {sb_glob:.4f}")
print(f"IC 95 % = [{b_glob - 1.96 * sb_glob:.4f} , {b_glob + 1.96 * sb_glob:.4f}]   n = {n_glob:,}")
print(f"a-value = {a_glob:.3f}")

dentro = abs(b_glob - 1.0) < 1.96 * sb_glob
conclusion(
    f"El b-value historico global es {b_glob:.3f} +- {sb_glob:.3f}. "
    + ("Es estadisticamente compatible con el valor canonico b = 1.0 "
       if dentro else
       f"Difiere significativamente del valor canonico b = 1.0 (el IC 95 % no lo contiene), lo "
       f"que refleja la mezcla de regimenes tectonicos de un catalogo global. ")
    + f"El ajuste lineal en escala semilogaritmica confirma que las magnitudes siguen la ley de "
      f"Gutenberg-Richter, es decir, una distribucion EXPONENCIAL. Esto valida el uso del "
      f"estimador de Aki y, sobre todo, habilita una segunda via de deteccion de anomalias: "
      f"comparar el b-value reciente de cada region contra su b-value historico mediante el TEST "
      f"DE UTSU (seccion 6.4). Una caida del b-value indica un desplazamiento del reparto de "
      f"energia hacia sismos de mayor tamano, aunque el numero total de eventos no cambie."
)

---
# 4. Fase 3 — Análisis bivariado y de correlación

Objetivo: identificar **estructuras de dependencia** entre variables y, sobre todo, entre las
variables independientes y el **target** (`n_eventos`).

In [ ]:
titulo("16. Matrices de correlacion: Pearson vs Spearman vs Kendall")

# Tabla a nivel de REGION-MES (unidad de analisis del target)
biv = panel_hist[["n_eventos", "mag_max", "mag_media", "prof_media", "log10_E_total"]].copy()
biv = biv[panel_hist["n_eventos"] > 0].dropna()

metodos = ["pearson", "spearman", "kendall"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
matrices = {}
for ax, m in zip(axes, metodos):
    M = biv.corr(method=m)
    matrices[m] = M
    sns.heatmap(M, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
                square=True, cbar=False, ax=ax, annot_kws={"size": 8})
    ax.set_title(f"{m.capitalize()}")
plt.suptitle("Matrices de correlacion sobre el panel region-mes (conjunto HISTORICO)",
             y=1.03, fontweight="bold")
plt.tight_layout()
plt.show()

comp = pd.DataFrame({
    "Pearson": matrices["pearson"]["n_eventos"],
    "Spearman": matrices["spearman"]["n_eventos"],
    "Kendall": matrices["kendall"]["n_eventos"],
}).drop(index="n_eventos").round(3)
comp["|Spearman| - |Pearson|"] = (comp["Spearman"].abs() - comp["Pearson"].abs()).round(3)
print("Correlacion de cada variable con el TARGET (n_eventos):")
display(comp)

difs = comp["|Spearman| - |Pearson|"].abs().max()
conclusion(
    f"Se reportan las tres medidas porque miden cosas distintas: PEARSON captura solo relaciones "
    f"LINEALES y exige normalidad aproximada (que la seccion 5 rechazara); SPEARMAN y KENDALL son "
    f"NO PARAMETRICAS, se basan en rangos y detectan cualquier relacion monotona, ademas de ser "
    f"robustas frente a los valores extremos que abundan en este dataset.\n"
    f"La discrepancia maxima entre |Spearman| y |Pearson| es de {difs:.3f}. "
    + ("Es sustancial, lo que confirma que las relaciones NO son lineales y que Pearson "
       "subestima la dependencia real: la medida valida aqui es Spearman."
       if difs > 0.10 else
       "Es pequena, de modo que las tres medidas coinciden en el orden de las asociaciones; aun "
       "asi se prefiere Spearman por su robustez ante las colas pesadas.")
    + f"\nKendall es sistematicamente menor en valor absoluto que Spearman por construccion "
      f"(mide concordancia de pares, no diferencias de rango al cuadrado); lo relevante es que "
      f"coincidan en SIGNO, como ocurre aqui."
)

In [ ]:
titulo("17. Relacion de las variables independientes con el TARGET")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
pares = [("mag_max", "Magnitud maxima del mes"),
         ("prof_media", "Profundidad media del mes (km)"),
         ("log10_E_total", "log10 de la energia total liberada")]
for ax, (v, etiqueta) in zip(axes, pares):
    ax.scatter(biv[v], biv["n_eventos"], s=7, alpha=0.15, color="#4C72B0")
    # tendencia por medianas moviles (robusta, no parametrica)
    q = pd.qcut(biv[v], 12, duplicates="drop")
    tend = biv.groupby(q, observed=True)["n_eventos"].median()
    centros_q = [iv.mid for iv in tend.index]
    ax.plot(centros_q, tend.values, color="#C44E52", lw=2.4, marker="o", ms=4,
            label="Mediana por decil")
    rho, p = stats.spearmanr(biv[v], biv["n_eventos"])
    ax.set_title(f"{etiqueta}\nSpearman rho = {rho:+.3f} (p = {p:.2e})")
    ax.set_xlabel(etiqueta)
    ax.set_ylabel("N.o de sismos en el mes")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

rho_e, _ = stats.spearmanr(biv["log10_E_total"], biv["n_eventos"])
rho_m, _ = stats.spearmanr(biv["mag_max"], biv["n_eventos"])
conclusion(
    f"La energia total del mes esta muy correlacionada con el numero de eventos "
    f"(rho = {rho_e:+.2f}), lo cual es en parte TAUTOLOGICO: mas sismos implican mas energia "
    f"acumulada. No aporta informacion independiente y por eso no se usara como criterio "
    f"separado de anomalia.\n"
    f"La relacion entre la magnitud maxima y el conteo (rho = {rho_m:+.2f}) es la interesante: "
    + ("es positiva, coherente con la ley de Gutenberg-Richter (para observar un sismo grande "
       "hace falta una muestra grande de sismos) y con el hecho de que un evento principal "
       "genera una secuencia de replicas que engrosa el conteo del mismo mes."
       if rho_m > 0 else
       "es debil o negativa, lo que sugiere que el tamano maximo y la frecuencia son dimensiones "
       "de la sismicidad relativamente independientes.")
    + "\nIMPLICACION: 'numero de eventos' y 'magnitud maxima' capturan facetas distintas del "
      "riesgo, por lo que el diagnostico final debe combinar AMBOS criterios (tasa y b-value) y "
      "no uno solo."
)

In [ ]:
titulo("18. Analisis bivariado por categorias (boxplots multiples)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Magnitud por rango de profundidad
sns.boxplot(data=base_uni, x="rango_prof", y="mag", ax=axes[0],
            color="#4C72B0", showfliers=False)
axes[0].set_title("Magnitud segun el rango de profundidad")
axes[0].set_xlabel("")
axes[0].set_ylabel("Magnitud (Mw)")
axes[0].tick_params(axis="x", labelrotation=12)

# (b) Magnitud en las 10 regiones mas activas
top10 = base_uni["region_nombre"].value_counts().head(10).index
sub = base_uni[base_uni["region_nombre"].isin(top10)]
orden = sub.groupby("region_nombre")["mag"].median().sort_values(ascending=False).index
sns.boxplot(
    data=sub,
    y="region_nombre",
    x="mag",
    order=orden,
    ax=axes[1],
    palette="viridis",
    showfliers=False
)
axes[1].set_title("Magnitud en las 10 regiones mas activas")
axes[1].set_xlabel("Magnitud (Mw)")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

# Contraste no parametrico: Kruskal-Wallis (ANOVA no exige normalidad -> se usa K-W)
grupos_prof = [g["mag"].dropna().values for _, g in base_uni.groupby("rango_prof", observed=True)
               if len(g) > 30]
H, p_kw = stats.kruskal(*grupos_prof)
print(f"Kruskal-Wallis (magnitud ~ rango de profundidad): H = {H:.2f}, p = {p_kw:.3e}")
print("  ", veredicto(p_kw, h0="las 3 poblaciones tienen la misma distribucion de magnitud",
                      h1="al menos un rango de profundidad difiere"))

grupos_reg = [g["mag"].dropna().values for _, g in sub.groupby("region_nombre", observed=True)]
H2, p_kw2 = stats.kruskal(*grupos_reg)
print(f"\nKruskal-Wallis (magnitud ~ region): H = {H2:.2f}, p = {p_kw2:.3e}")
print("  ", veredicto(p_kw2, h0="todas las regiones tienen la misma distribucion de magnitud",
                      h1="al menos una region difiere"))

conclusion(
    f"Se emplea KRUSKAL-WALLIS y no un ANOVA porque el ANOVA exige normalidad y homocedasticidad, "
    f"supuestos que la seccion 5 rechaza de forma contundente. Kruskal-Wallis solo requiere "
    f"independencia y trabaja sobre rangos.\n"
    + (f"El test region (p = {p_kw2:.1e}) confirma que la distribucion de magnitudes DIFIERE "
       f"significativamente entre regiones. Esta es la justificacion estadistica central del "
       f"diseno del estudio: no existe un 'comportamiento sismico normal' unico y global; cada "
       f"region tiene su propia linea base, y por eso la deteccion de anomalias debe hacerse "
       f"REGION POR REGION y no contra un promedio mundial."
       if p_kw2 < ALPHA else
       "Las regiones no difieren significativamente en magnitud, de modo que las diferencias "
       "entre ellas se concentran en la TASA de ocurrencia mas que en el tamano de los eventos.")
)

In [ ]:
titulo("19. Asociacion entre variables CATEGORICAS: chi-cuadrado y V de Cramer")


def v_cramer(tabla):
    chi2 = stats.chi2_contingency(tabla)[0]
    n = tabla.values.sum()
    r, k = tabla.shape
    phi2 = chi2 / n
    # correccion de sesgo de Bergsma (2013)
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2c / max(min(kc - 1, rc - 1), 1e-12))


tabla_ct = pd.crosstab(base_uni["rango_prof"], base_uni["magType"])
tabla_ct = tabla_ct.loc[:, tabla_ct.sum() > 100]
chi2, p_chi, gl, esperadas = stats.chi2_contingency(tabla_ct)

print("Tabla de contingencia (rango de profundidad x tipo de magnitud):")
display(tabla_ct)
print(f"\nChi-cuadrado = {chi2:,.1f}  |  gl = {gl}  |  p = {p_chi:.3e}")
print(f"V de Cramer  = {v_cramer(tabla_ct):.3f}   (0 = independencia, 1 = asociacion perfecta)")
print("  ", veredicto(p_chi, h0="profundidad y tipo de magnitud son independientes",
                      h1="existe asociacion entre ambas"))

# Residuos estandarizados: donde se concentra la dependencia
resid = (tabla_ct - esperadas) / np.sqrt(esperadas)
fig, ax = plt.subplots(figsize=(9, 3.2))
sns.heatmap(resid, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Residuos estandarizados de Pearson (|z| > 2 indica celda influyente)")
plt.tight_layout()
plt.show()

V = v_cramer(tabla_ct)
conclusion(
    f"Con n = {tabla_ct.values.sum():,} observaciones, el test chi-cuadrado detecta como "
    f"significativa casi cualquier diferencia, por lo que el p-valor por si solo no informa: hay "
    f"que leer el TAMANO DEL EFECTO.\n"
    f"La V de Cramer = {V:.3f} indica una asociacion "
    + ("FUERTE entre el rango de profundidad y la escala de magnitud empleada."
       if V > 0.35 else
       "MODERADA entre el rango de profundidad y la escala de magnitud empleada."
       if V > 0.15 else
       "DEBIL: aunque el test resulte significativo, la dependencia practica es despreciable.")
    + f"\nLos residuos estandarizados de Pearson localizan donde se concentra la dependencia "
      f"(|z| > 2 senala celdas influyentes): si ciertas escalas de magnitud se emplean "
      f"preferentemente en determinados rangos de profundidad, existe una heterogeneidad de "
      f"PROCEDIMIENTO del catalogo, no de la sismicidad, que conviene tener presente al comparar "
      f"epocas distintas."
)

In [ ]:
titulo("20. Estructura TEMPORAL del target: autocorrelacion")

serie_global = panel.groupby("mes")["n_eventos"].sum()
serie_global.index = serie_global.index.to_timestamp()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7))
ax1.plot(serie_global.index, serie_global.values, lw=0.9, color="#4C72B0")
ax1.plot(serie_global.index, serie_global.rolling(12, center=True).mean(),
         lw=2.2, color="#C44E52", label="Media movil de 12 meses")
ax1.axvline(MES_CORTE.to_timestamp(), color="black", ls="--", lw=1.6,
            label=f"corte historico/reciente ({MES_CORTE})")
ax1.set_title("Serie mensual global de sismos M>=5.0")
ax1.set_ylabel("N.o de sismos/mes")
ax1.legend(fontsize=8)


def autocorr(x, nlags=36):
    x = np.asarray(x, dtype=float) - np.mean(x)
    denom = np.sum(x ** 2)
    return np.array([1.0] + [np.sum(x[k:] * x[:-k]) / denom for k in range(1, nlags + 1)])


serie_h = panel_hist.groupby("mes")["n_eventos"].sum()
ac = autocorr(serie_h.values, 36)
ic95 = 1.96 / np.sqrt(len(serie_h))
ax2.bar(range(len(ac)), ac, color="#4C72B0", width=0.7)
ax2.axhline(ic95, color="#C44E52", ls="--", lw=1.2, label="banda IC 95 % (ruido blanco)")
ax2.axhline(-ic95, color="#C44E52", ls="--", lw=1.2)
ax2.axhline(0, color="black", lw=0.8)
ax2.set_title("Funcion de autocorrelacion (FAC) del conteo mensual — conjunto HISTORICO")
ax2.set_xlabel("Rezago (meses)")
ax2.set_ylabel("Autocorrelacion")
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

signif = int(np.sum(np.abs(ac[1:13]) > ic95))
print(f"Rezagos 1-12 con autocorrelacion significativa: {signif} de 12  (banda = +-{ic95:.3f})")
print(f"Autocorrelacion de rezago 1: {ac[1]:+.3f}")

conclusion(
    f"La FAC muestra {signif} de los primeros 12 rezagos fuera de la banda de ruido blanco. "
    + (f"Existe MEMORIA en la serie: un mes activo tiende a ser seguido por otro mes activo. Esto "
       f"contradice el supuesto de INDEPENDENCIA del proceso de Poisson y es la manifestacion "
       f"temporal del agrupamiento por replicas ya detectado en la sobredispersion.\n"
       f"CONSECUENCIA: los p-valores de un test de Poisson puro serian demasiado optimistas "
       f"(anticonservadores). Por eso la seccion 6 aplica una correccion cuasi-Poisson que infla "
       f"la varianza por el factor de dispersion estimado."
       if signif >= 2 else
       "La serie es practicamente ruido blanco a escala mensual, lo que respalda el supuesto de "
       "independencia entre meses del modelo de Poisson.")
    + "\nAdemas, la media movil de 12 meses permite verificar visualmente que no hay una tendencia "
      "global creciente artificial en la ventana de trabajo, condicion necesaria para que la "
      "comparacion historico/reciente sea legitima."
)

---
# 5. Fase 4 — Diagnóstico inferencial y validación de supuestos



## 5.1 Pruebas de normalidad



In [ ]:
titulo("21. Bateria de pruebas de normalidad")


def bateria_normalidad(serie, nombre, n_shapiro=45, n_rep=200):
    """Shapiro-Wilk sobre submuestras pequenas + D'Agostino + K-S(Lilliefors) + Anderson-Darling."""
    s = pd.Series(serie).dropna().astype(float)
    n = len(s)
    res = {"Variable": nombre, "n": n}

    # --- Shapiro-Wilk sobre submuestras de n<50 (repetido para estabilidad) ---
    if n >= n_shapiro:
        ps = [stats.shapiro(s.sample(n_shapiro, random_state=i)).pvalue for i in range(n_rep)]
        res["Shapiro p (mediana de %d submuestras n=%d)" % (n_rep, n_shapiro)] = float(np.median(ps))
        res["% submuestras que rechazan"] = float(np.mean(np.array(ps) < ALPHA) * 100)
    else:
        res["Shapiro p (mediana de %d submuestras n=%d)" % (n_rep, n_shapiro)] = stats.shapiro(s).pvalue
        res["% submuestras que rechazan"] = np.nan

    # --- D'Agostino-Pearson K^2 -------------------------------------------
    res["DAgostino p"] = stats.normaltest(s).pvalue if n >= 20 else np.nan

    # --- K-S tipo Lilliefors (parametros estimados de la muestra) ----------
    z = (s - s.mean()) / s.std(ddof=1)
    res["KS p"] = stats.kstest(z, "norm").pvalue

    # --- Anderson-Darling (critico al 5 %) ---------------------------------
    ad = stats.anderson(s.sample(min(n, 20000), random_state=RANDOM_STATE), dist="norm")
    res["AD estadistico"] = float(ad.statistic)
    res["AD critico 5%"] = float(ad.critical_values[2])
    res["AD rechaza"] = bool(ad.statistic > ad.critical_values[2])
    res["Asimetria"] = float(stats.skew(s))
    res["Curtosis"] = float(stats.kurtosis(s))
    return res


series_test = {
    "mag": eventos_hist["mag"],
    "depth": eventos_hist["depth"],
    "log10_E": eventos_hist["log10_E"],
    "n_eventos (TARGET)": panel_hist["n_eventos"],
    "conteo mensual global": panel_hist.groupby("mes")["n_eventos"].sum(),
}
tabla_norm = pd.DataFrame([bateria_normalidad(v, k) for k, v in series_test.items()])
display(tabla_norm.set_index("Variable").T)

n_rechaza = int(tabla_norm["AD rechaza"].sum())
conclusion(
    f"{n_rechaza} de {len(tabla_norm)} variables rechazan la normalidad segun Anderson-Darling, y "
    f"las cuatro pruebas coinciden en el diagnostico. El resultado no es un problema del dato: es "
    f"una propiedad del fenomeno. La magnitud es exponencial (Gutenberg-Richter), la profundidad "
    f"es multimodal y el target es un CONTEO discreto con exceso de ceros.\n"
    f"CONSECUENCIAS PARA EL DISENO DEL ANALISIS:\n"
    f"1) Queda descartada toda tecnica que asuma normalidad: intervalos t, ANOVA, regresion lineal "
    f"ordinaria sobre los conteos crudos y limites 'media +- 3 sigma' calculados con la desviacion "
    f"estandar muestral.\n"
    f"2) La deteccion de anomalias debe apoyarse en la distribucion CORRECTA del fenomeno: la "
    f"POISSON para los conteos y la EXPONENCIAL para las magnitudes.\n"
    f"3) Para comparar grupos se usara estadistica NO PARAMETRICA (Mann-Whitney, Kruskal-Wallis, "
    f"Kolmogorov-Smirnov de dos muestras)."
)

In [ ]:
titulo("22. Q-Q plots: diagnostico visual de normalidad")

fig, axes = plt.subplots(1, 4, figsize=(16.5, 4.2))
for ax, (nombre, serie) in zip(axes, list(series_test.items())[:4]):
    s = pd.Series(serie).dropna().astype(float)
    s = s.sample(min(len(s), 5000), random_state=RANDOM_STATE)
    stats.probplot(s, dist="norm", plot=ax)
    ax.set_title(f"Q-Q normal: {nombre}", fontsize=10)
    ax.get_lines()[0].set(markersize=2.5, alpha=0.45, color="#4C72B0")
    ax.get_lines()[1].set(color="#C44E52", lw=1.8)
    ax.set_xlabel("Cuantiles teoricos")
    ax.set_ylabel("Cuantiles muestrales")
plt.tight_layout()
plt.show()

conclusion(
    "Los Q-Q plots traducen el rechazo numerico a un diagnostico interpretable: si los puntos se "
    "separan de la recta roja en el extremo SUPERIOR, la variable tiene cola derecha pesada "
    "(mas valores extremos altos de los que predice la normal); la forma escalonada del target "
    "delata su naturaleza DISCRETA, que ninguna distribucion continua puede reproducir. "
    "Visualizar esto es obligatorio: un p-valor solo dice 'no es normal', el Q-Q plot dice EN QUE "
    "SENTIDO no lo es, que es lo que determina la transformacion o el modelo alternativo a usar."
)

## 5.2 ¿Se cumple el supuesto de Poisson? Índice de dispersión



In [ ]:
titulo("23. Indice de dispersion por region (conjunto HISTORICO)")


def indice_dispersion(conteos):
    """Test chi-cuadrado de equidispersion de Poisson."""
    x = np.asarray(conteos, dtype=float)
    n = len(x)
    media = x.mean()
    if n < 10 or media <= 0:
        return np.nan, np.nan, np.nan
    D = (n - 1) * x.var(ddof=1) / media
    p = 1 - stats.chi2.cdf(D, df=n - 1)      # unilateral: sobredispersion
    return D, p, x.var(ddof=1) / media       # D, p-valor, phi


filas = []
for reg, g in panel_hist.groupby("region"):
    D, p, phi = indice_dispersion(g["n_eventos"].values)
    filas.append({"region": reg,
                  "region_nombre": g["region_nombre"].iloc[0],
                  "media_hist": g["n_eventos"].mean(),
                  "var_hist": g["n_eventos"].var(ddof=1),
                  "phi": phi, "D": D, "p_dispersion": p})

dispersion = pd.DataFrame(filas).dropna(subset=["phi"])
dispersion["sobredisperso"] = dispersion["p_dispersion"] < ALPHA

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
sns.histplot(dispersion["phi"], bins=40, color="#4C72B0", ax=axes[0])
axes[0].axvline(1, color="#C44E52", ls="--", lw=2, label="phi = 1 (Poisson pura)")
axes[0].set_title("Distribucion del factor de dispersion phi entre regiones")
axes[0].set_xlabel("phi = varianza / media")
axes[0].legend()

axes[1].scatter(dispersion["media_hist"], dispersion["var_hist"], s=22, alpha=0.7,
                color="#4C72B0")
lim = np.linspace(0, dispersion["media_hist"].max() * 1.05, 50)
axes[1].plot(lim, lim, color="#C44E52", lw=2, label="Var = Media (Poisson)")
axes[1].set_xlabel("Media de conteos mensuales")
axes[1].set_ylabel("Varianza de conteos mensuales")
axes[1].set_title("Relacion media-varianza por region")
axes[1].legend()
plt.tight_layout()
plt.show()

pct_sobre = dispersion["sobredisperso"].mean() * 100
PHI_MEDIANO = float(dispersion["phi"].median())
print(f"Regiones sobredispersas (p < {ALPHA}): {dispersion['sobredisperso'].sum()} de "
      f"{len(dispersion)} ({pct_sobre:.1f} %)")
print(f"phi mediano = {PHI_MEDIANO:.3f}   |   phi P90 = {dispersion['phi'].quantile(0.90):.3f}")
display(dispersion.sort_values("phi", ascending=False)
        .head(8)[["region_nombre", "region", "media_hist", "var_hist", "phi", "p_dispersion"]]
        .round(3))

conclusion(
    f"El {pct_sobre:.0f} % de las regiones rechaza la equidispersion, con phi mediano = "
    f"{PHI_MEDIANO:.2f}. La nube media-varianza se situa por ENCIMA de la recta identidad, que es "
    f"la firma grafica de la sobredispersion.\n"
    f"INTERPRETACION FISICA: los sismos NO son independientes entre si. Un evento principal "
    f"desencadena una secuencia de replicas (ley de Omori: la tasa de replicas decae como 1/t), de "
    f"modo que los eventos llegan agrupados y la varianza excede a la media.\n"
    f"CONSECUENCIA OPERATIVA, y es la decision estadistica mas importante de todo el analisis: si "
    f"se aplicara un test de Poisson puro, su varianza estaria subestimada por un factor de "
    f"~{PHI_MEDIANO:.1f}, los intervalos serian demasiado estrechos y se declararian anomalas "
    f"muchas regiones normales (inflacion del error de tipo I). Por eso en la seccion 6 se reporta "
    f"tanto el test exacto de Poisson como una version CUASI-POISSON que corrige la varianza por "
    f"phi, y se toma esta ultima como criterio principal."
)

## 5.3 ¿Son exponenciales los tiempos entre eventos?



In [ ]:
titulo("24. Test K-S de exponencialidad de los tiempos entre eventos")

top_regiones = (panel_hist.groupby(["region", "region_nombre"])["n_eventos"].sum()
                .sort_values(ascending=False).head(6).reset_index())

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
res_exp = []
for ax, (_, fila) in zip(axes.ravel(), top_regiones.iterrows()):
    reg = fila["region"]
    d = dfm[(dfm["region"] == reg) & (dfm["periodo"] == "HISTORICO")].sort_values("time")
    t = d["time"].diff().dt.total_seconds().dropna() / 3600.0
    t = t[t > 0]
    if len(t) < 50:
        continue
    lam = 1.0 / t.mean()
    ks, p_ks = stats.kstest(t, "expon", args=(0, 1 / lam))
    cv = t.std() / t.mean()
    res_exp.append({"region": reg, "region_nombre": fila["region_nombre"], "n": len(t),
                    "CV": cv, "KS": ks, "p_KS": p_ks,
                    "media_dt_h": t.mean(), "mediana_dt_h": t.median()})

    x = np.sort(t)
    ax.plot(x, np.arange(1, len(x) + 1) / len(x), lw=1.8, color="#4C72B0",
            label="CDF empirica")
    ax.plot(x, stats.expon.cdf(x, 0, 1 / lam), lw=1.8, ls="--", color="#C44E52",
            label="Exponencial teorica")
    ax.set_xscale("log")
    ax.set_title(f"{fila['region_nombre']} [{reg}]\nCV = {cv:.2f} | KS p = {p_ks:.1e}",
                 fontsize=9.5)
    ax.set_xlabel("Tiempo entre sismos (h, log)")
    ax.set_ylabel("Probabilidad acumulada")
    ax.legend(fontsize=7.5)
plt.suptitle("Contraste del supuesto de Poisson: exponencialidad de los tiempos entre eventos",
             y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()

tabla_exp = pd.DataFrame(res_exp).round(4)
display(tabla_exp)

cv_medio = tabla_exp["CV"].mean()
n_rech = int((tabla_exp["p_KS"] < ALPHA).sum())
conclusion(
    f"{n_rech} de {len(tabla_exp)} regiones rechazan la exponencialidad (K-S). El coeficiente de "
    f"variacion medio es CV = {cv_medio:.2f}, frente al valor CV = 1 que exige la exponencial.\n"
    + (f"CV > 1 confirma un proceso AGRUPADO (clustered), no poissoniano puro: hay muchos "
       f"intervalos muy cortos (replicas que siguen a un sismo principal) y tambien periodos de "
       f"calma largos. La curva empirica se situa por encima de la teorica en los tiempos cortos, "
       f"que es exactamente la firma de las secuencias de replicas."
       if cv_medio > 1.15 else
       f"CV cercano a 1: a escala regional el proceso es razonablemente compatible con Poisson.")
    + f"\nEste resultado, junto con la sobredispersion y la autocorrelacion positiva, cierra un "
      f"diagnostico coherente: el proceso sismico tiene MEMORIA A CORTO PLAZO. El modelo de "
      f"Poisson sigue siendo la referencia correcta para la sismicidad DE FONDO a escala mensual, "
      f"pero sus intervalos deben ampliarse por el factor phi antes de declarar una anomalia."
)

## 5.4 Homogeneidad histórico vs. reciente: control de artefactos




In [ ]:
titulo("25. Contraste de homogeneidad entre periodos")

m_h = eventos_hist["mag"].dropna()
m_r = eventos_rec["mag"].dropna()

U, p_mw = stats.mannwhitneyu(m_h, m_r, alternative="two-sided")
ks2, p_ks2 = stats.ks_2samp(m_h, m_r)
lev, p_lev = stats.levene(m_h, m_r, center="median")     # Brown-Forsythe

print("MAGNITUD: historico vs reciente")
print(f"  Mann-Whitney U : U = {U:,.0f} | {veredicto(p_mw, h0='misma distribucion de magnitud', h1='las medianas difieren')}")
print(f"  K-S 2 muestras : D = {ks2:.4f} | {veredicto(p_ks2, h0='misma distribucion', h1='las distribuciones difieren')}")
print(f"  Levene (B-F)   : W = {lev:.2f} | {veredicto(p_lev, h0='misma varianza', h1='las varianzas difieren (heterocedasticidad)')}")

# Tamano del efecto (delta de Cliff via el estadistico U): magnitud practica del cambio
delta_cliff = 2 * U / (len(m_h) * len(m_r)) - 1
print(f"  Tamano del efecto (delta de Cliff) = {delta_cliff:+.4f}  "
      f"(|d|<0.15 despreciable, <0.33 pequeno, <0.47 mediano, si no grande)")

# Composicion del catalogo por periodo (control de artefactos)
ct_tipo = pd.crosstab(dfm["periodo"], dfm["magType"])
ct_tipo = ct_tipo.loc[:, ct_tipo.sum() > 200]
chi2_t, p_tipo, _, _ = stats.chi2_contingency(ct_tipo)
print(f"\nCOMPOSICION DE 'magType' por periodo: chi2 = {chi2_t:,.1f} | "
      f"{veredicto(p_tipo, h0='misma composicion de escalas', h1='la composicion del catalogo CAMBIO')}")
display(ct_tipo.div(ct_tipo.sum(axis=1), axis=0).mul(100).round(2))

fig, ax = plt.subplots(figsize=(9.5, 4.4))
for etiqueta, serie, color in [("HISTORICO", m_h, "#4C72B0"), ("RECIENTE", m_r, "#C44E52")]:
    sns.kdeplot(serie, ax=ax, label=f"{etiqueta} (n={len(serie):,})", lw=2.2,
                color=color, fill=True, alpha=0.18, clip=(5, 9.6))
ax.set_title("Distribucion de la magnitud: HISTORICO vs RECIENTE")
ax.set_xlabel("Magnitud (Mw)")
ax.legend()
plt.tight_layout()
plt.show()

conclusion(
    f"El tamano del efecto sobre la magnitud es delta de Cliff = {delta_cliff:+.3f}, es decir, un "
    f"efecto "
    + ("DESPRECIABLE" if abs(delta_cliff) < 0.15 else
       "PEQUENO" if abs(delta_cliff) < 0.33 else
       "MEDIANO" if abs(delta_cliff) < 0.47 else "GRANDE")
    + f". Con n del orden de 10^5, el p-valor de Mann-Whitney es casi irrelevante: lo que decide "
      f"es el tamano del efecto.\n"
    + (f"ADVERTENCIA: la composicion de escalas de magnitud (magType) SI cambio entre periodos "
       f"(p = {p_tipo:.1e}). Esto es un artefacto de procedimiento del catalogo, no sismicidad. "
       f"Por eso el criterio principal de anomalia sera la TASA DE OCURRENCIA (conteo de eventos "
       f"por encima de un Mc fijo de 5.0), que es mucho mas robusta a un cambio de escala que "
       f"cualquier estadistico basado en el valor numerico de la magnitud."
       if p_tipo < ALPHA else
       f"La composicion del catalogo por escalas de magnitud es estable entre periodos, de modo "
       f"que las diferencias que se detecten seran atribuibles a la sismicidad y no a un cambio "
       f"de procedimiento.")
)

## 5.5 Detección de observaciones anómalas e influyentes



In [ ]:
titulo("26. Deteccion de observaciones anomalas en la serie mensual global")

serie_mes = panel.groupby("mes")["n_eventos"].sum()
x = serie_mes.values.astype(float)
fechas = serie_mes.index.to_timestamp()

# --- (a) Tukey ---------------------------------------------------------------
q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1
lim_tukey = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
out_tukey = (x < lim_tukey[0]) | (x > lim_tukey[1])

# --- (b) Z robusto basado en la MAD ------------------------------------------
mediana = np.median(x)
mad = np.median(np.abs(x - mediana))
z_rob = 0.6745 * (x - mediana) / mad if mad > 0 else np.zeros_like(x)
out_mad = np.abs(z_rob) > 3.5

# --- (c) Grubbs (referencia; exige normalidad) -------------------------------
n = len(x)
G = np.max(np.abs(x - x.mean())) / x.std(ddof=1)
t_crit = stats.t.ppf(1 - ALPHA / (2 * n), n - 2)
G_crit = (n - 1) / np.sqrt(n) * np.sqrt(t_crit ** 2 / (n - 2 + t_crit ** 2))

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.plot(fechas, x, lw=1.0, color="#4C72B0", label="Conteo mensual global")
ax.scatter(fechas[out_mad], x[out_mad], s=52, color="#C44E52", zorder=5,
           label=f"Anomalo por Z robusto/MAD (n={out_mad.sum()})")
ax.scatter(fechas[out_tukey & ~out_mad], x[out_tukey & ~out_mad], s=30,
           facecolors="none", edgecolors="#DD8452", lw=1.5, zorder=4,
           label=f"Solo por Tukey (n={(out_tukey & ~out_mad).sum()})")
ax.axhline(lim_tukey[1], color="#DD8452", ls=":", lw=1.4, label="Limite superior de Tukey")
ax.axvline(MES_CORTE.to_timestamp(), color="black", ls="--", lw=1.4, label="corte 80/20")
ax.set_title("Meses atipicos en la actividad sismica global")
ax.set_xlabel("Fecha")
ax.set_ylabel("N.o de sismos M>=5.0")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print(f"Tukey       : {out_tukey.sum()} meses atipicos, limites = "
      f"[{lim_tukey[0]:.0f} , {lim_tukey[1]:.0f}]")
print(f"Z robusto   : {out_mad.sum()} meses atipicos (umbral |Z|>3.5, MAD = {mad:.1f})")
print(f"Grubbs      : G = {G:.3f} vs G_critico = {G_crit:.3f}  ->  "
      + ("se detecta al menos un valor extremo" if G > G_crit else "no se detecta valor extremo"))

top_meses = serie_mes.iloc[np.argsort(-x)[:5]]
print("\nMeses con mayor actividad global:")
for m, v in top_meses.items():
    print(f"   {m}: {int(v):,} sismos   (Z robusto = {0.6745 * (v - mediana) / mad:+.2f})")

conclusion(
    (f"Los tres criterios coinciden en senalar los mismos meses extremos. "
     if out_mad.sum() else
     f"Ningun mes supera los umbrales de atipicidad: la serie global no contiene valores "
     f"extremos aislados, sino variacion continua. ")
    + f"Los criterios NO son intercambiables: "
    f"Tukey supone simetria (que la seccion 5.1 rechazo) y Grubbs supone normalidad, por lo que "
    f"aqui solo valen como referencia. El criterio valido es el Z ROBUSTO basado en la MAD, que "
    f"no se ve arrastrado por los propios valores extremos que pretende detectar (evita el "
    f"enmascaramiento).\n"
    f"DECISION CRITICA: estos meses NO se eliminan. En un analisis de riesgo sismico, los meses "
    f"extremos corresponden a grandes terremotos con sus secuencias de replicas; son la SENAL que "
    f"buscamos, no ruido. Eliminarlos equivaldria a borrar exactamente los eventos que el estudio "
    f"debe detectar. Se documentan, se vigila que no distorsionen la linea base historica (por eso "
    f"lambda se estima como tasa media sobre un periodo largo de ~{M_HIST // 12} anios, que "
    f"amortigua su influencia) y se conservan."
)

---
# 6. Fase 5 — NÚCLEO: ¿qué regiones están fuera de lo esperado?



In [ ]:
titulo("27. Deriva global del catalogo y factor de estandarizacion")

X_h_glob = int(panel_hist["n_eventos"].sum())
X_r_glob = int(panel_rec["n_eventos"].sum())
tasa_h_glob = X_h_glob / M_HIST      # sismos/mes en todo el catalogo (historico)
tasa_r_glob = X_r_glob / M_REC       # sismos/mes en todo el catalogo (reciente)
F_GLOBAL = tasa_r_glob / tasa_h_glob

# Test exacto de la deriva global (binomial condicional)
p0_glob = M_REC / (M_HIST + M_REC)
bt_glob = stats.binomtest(X_r_glob, X_h_glob + X_r_glob, p0_glob, alternative="two-sided")

print(f"Tasa global HISTORICA : {tasa_h_glob:8.2f} sismos/mes  ({X_h_glob:,} en {M_HIST} meses)")
print(f"Tasa global RECIENTE  : {tasa_r_glob:8.2f} sismos/mes  ({X_r_glob:,} en {M_REC} meses)")
print(f"Factor de deriva global f = {F_GLOBAL:.4f}  "
      f"({(F_GLOBAL - 1) * 100:+.1f} % respecto al historico)")
print("  ", veredicto(bt_glob.pvalue, h0="la tasa global no cambio",
                      h1="la tasa global del catalogo SI cambio"))

conclusion(
    f"La tasa global del catalogo es {F_GLOBAL:.2f} veces la historica "
    f"({(F_GLOBAL - 1) * 100:+.1f} %). "
    + (f"El cambio es SIGNIFICATIVO, de modo que existe una deriva de fondo comun a todo el "
       f"catalogo. Comparar cada region contra su propio pasado SIN corregir esta deriva "
       f"produciria un resultado absurdo: casi todas las regiones apareceran 'anomalas', porque "
       f"todas comparten el mismo aumento de cobertura instrumental.\n"
       f"DECISION: se adopta la ESTANDARIZACION INDIRECTA. El valor esperado de cada region se "
       f"multiplica por f = {F_GLOBAL:.3f}, y la pregunta pasa a ser si la region cambio MAS QUE "
       f"EL CONJUNTO. Se reportan las dos versiones (bruta y estandarizada) para que la "
       f"diferencia sea auditable, pero el criterio principal es la ESTANDARIZADA."
       if bt_glob.pvalue < ALPHA else
       f"El cambio global NO es significativo: el catalogo es estable entre periodos. La "
       f"estandarizacion (f ~ 1) apenas modificara los resultados, pero se aplica igualmente por "
       f"consistencia metodologica.")
)

In [ ]:
titulo("28. Contrastes de tasa por region: Poisson exacto y cuasi-Poisson")


def bh_fdr(pvals):
    """Correccion de Benjamini-Hochberg: devuelve los q-valores (FDR ajustada)."""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    orden = np.argsort(p)
    q = np.empty(n)
    minimo = 1.0
    for rango in range(n - 1, -1, -1):
        idx = orden[rango]
        minimo = min(minimo, p[idx] * n / (rango + 1))
        q[idx] = minimo
    return np.clip(q, 0, 1)


phi_por_region = dispersion.set_index("region")["phi"].to_dict()
resultados = []

for reg, g in panel.groupby("region"):
    gh = g[g["periodo"] == "HISTORICO"]["n_eventos"]
    gr = g[g["periodo"] == "RECIENTE"]["n_eventos"]
    Th, Tr = len(gh), len(gr)                 # exposicion en meses
    Xh, Xr = int(gh.sum()), int(gr.sum())
    if Xh + Xr < 10:                          # muestra insuficiente para inferencia
        continue

    lam_h = Xh / Th                           # tasa historica (sismos/mes)
    lam_r = Xr / Tr                           # tasa reciente
    esperado_bruto = lam_h * Tr               # esperado sin corregir la deriva global
    esperado = esperado_bruto * F_GLOBAL      # esperado ESTANDARIZADO

    # --- (1) Test EXACTO condicional de dos tasas de Poisson (estandarizado) ---
    # Bajo H0, Xr | (Xh+Xr) ~ Binomial(Xh+Xr, p0), con la exposicion reciente
    # ponderada por el factor de deriva global f.
    p0 = (Tr * F_GLOBAL) / (Th + Tr * F_GLOBAL)
    bt = stats.binomtest(Xr, Xh + Xr, p0, alternative="two-sided")
    p_exacto = bt.pvalue
    lo, hi = bt.proportion_ci(confidence_level=0.95, method="exact")
    # IC de la razon de tasas ESTANDARIZADA (RTE)
    rte_lo = (lo / (1 - lo)) * (Th / (Tr * F_GLOBAL)) if lo < 1 else np.nan
    rte_hi = (hi / (1 - hi)) * (Th / (Tr * F_GLOBAL)) if hi < 1 else np.inf

    # --- (2) Z CUASI-POISSON: varianza inflada por el factor de dispersion -----
    phi = max(phi_por_region.get(reg, 1.0), 1.0)
    z_q = (Xr - esperado) / np.sqrt(phi * esperado) if esperado > 0 else np.nan
    p_quasi = 2 * (1 - stats.norm.cdf(abs(z_q))) if np.isfinite(z_q) else np.nan

    # --- (3) Versiones SIN estandarizar, solo para cuantificar el efecto ------
    z_bruto = ((Xr - esperado_bruto) / np.sqrt(phi * esperado_bruto)
               if esperado_bruto > 0 else np.nan)
    p_bruto = 2 * (1 - stats.norm.cdf(abs(z_bruto))) if np.isfinite(z_bruto) else np.nan
    # Poisson puro sin estandarizar: el criterio "ingenuo" que se quiere evitar
    z_ing = ((Xr - esperado_bruto) / np.sqrt(esperado_bruto)
             if esperado_bruto > 0 else np.nan)
    p_ingenuo = 2 * (1 - stats.norm.cdf(abs(z_ing))) if np.isfinite(z_ing) else np.nan

    resultados.append({
        "region": reg,
        "region_nombre": g["region_nombre"].iloc[0],
        "meses_hist": Th, "meses_rec": Tr,
        "n_hist": Xh, "n_rec": Xr,
        "tasa_hist": lam_h, "tasa_rec": lam_r,
        "esperado_rec": esperado,
        "exceso": Xr - esperado,
        "razon_tasas": lam_r / lam_h if lam_h > 0 else np.nan,      # bruta
        "RTE": Xr / esperado if esperado > 0 else np.nan,           # estandarizada
        "RTE_IC_inf": rte_lo, "RTE_IC_sup": rte_hi,
        "phi": phi,
        "p_poisson": p_exacto,
        "z_quasi": z_q, "p_quasi": p_quasi,
        "p_bruto": p_bruto,
        "p_ingenuo": p_ingenuo,
    })

res = pd.DataFrame(resultados)
res["q_poisson"] = bh_fdr(res["p_poisson"])
res["q_quasi"] = bh_fdr(res["p_quasi"].fillna(1.0))
res["q_bruto"] = bh_fdr(res["p_bruto"].fillna(1.0))
res["signif_poisson"] = res["q_poisson"] < ALPHA
res["signif_quasi"] = res["q_quasi"] < ALPHA
res["signif_bruto"] = res["q_bruto"] < ALPHA
res["direccion"] = np.where(res["RTE"] > 1, "AUMENTO", "DESCENSO")

res["q_ingenuo"] = bh_fdr(res["p_ingenuo"].fillna(1.0))

comparativa = pd.DataFrame({
    "Criterio": ["Poisson puro, SIN estandarizar (ingenuo)",
                 "Cuasi-Poisson, SIN estandarizar",
                 "Poisson exacto ESTANDARIZADO",
                 "Cuasi-Poisson ESTANDARIZADO  <-- PRINCIPAL"],
    "Corrige sobredispersion": ["no", "si", "no", "si"],
    "Corrige deriva del catalogo": ["no", "no", "si", "si"],
    "Regiones significativas": [int((res["q_ingenuo"] < ALPHA).sum()),
                                int(res["signif_bruto"].sum()),
                                int(res["signif_poisson"].sum()),
                                int(res["signif_quasi"].sum())],
})
comparativa["% del total"] = (comparativa["Regiones significativas"] / len(res) * 100).round(1)
print(f"Regiones evaluadas: {len(res)}\n")
display(comparativa)

print(f"\nSin correccion por multiplicidad (p<{ALPHA} en bruto): "
      f"{(res['p_quasi'] < ALPHA).sum()} regiones; de ellas, ~{ALPHA * len(res):.0f} serian "
      f"falsos positivos esperados solo por azar.")

conclusion(
    f"La tabla comparativa cuantifica el efecto de cada correccion, y esa es su utilidad "
    f"principal. El criterio ingenuo (Poisson puro sin estandarizar) senala "
    f"{comparativa.iloc[0]['Regiones significativas']} regiones de {len(res)}; el criterio "
    f"correcto (cuasi-Poisson estandarizado) senala {res['signif_quasi'].sum()}.\n"
    f"Las dos correcciones actuan sobre errores distintos: la ESTANDARIZACION elimina la deriva "
    f"comun del catalogo (un sesgo sistematico compartido por todas las regiones) y la CUASI-"
    f"POISSON amplia la varianza por el agrupamiento en replicas (un exceso de dispersion). "
    f"Ambas van en la misma direccion: reducir falsos positivos.\n"
    f"RESULTADO: {res['signif_quasi'].sum()} de {len(res)} regiones "
    f"({res['signif_quasi'].mean() * 100:.1f} %) presentan una actividad reciente "
    f"estadisticamente fuera de lo esperado, una vez descontadas la deriva instrumental y la "
    f"dependencia entre eventos, con control de FDR al {ALPHA:.0%}."
)

In [ ]:
titulo("29. Ranking de regiones fuera de lo esperado")

ranking = (res[res["signif_quasi"]]
           .sort_values("z_quasi", key=abs, ascending=False)
           .loc[:, ["region_nombre", "region", "n_hist", "tasa_hist", "n_rec", "tasa_rec",
                    "esperado_rec", "exceso", "RTE", "RTE_IC_inf", "RTE_IC_sup",
                    "phi", "z_quasi", "q_quasi", "direccion"]]
           .round(4))
print(f"Regiones estadisticamente FUERA DE LO ESPERADO: {len(ranking)} de {len(res)}")
print("RTE = razon de tasas estandarizada = observado / esperado (1.0 = exactamente lo esperado)")
display(ranking.head(20))

if len(ranking):
    top = ranking.reindex(ranking["z_quasi"].abs().sort_values(ascending=False).index).head(15)
    top = top.iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(top))))
    colores = ["#C44E52" if d == "AUMENTO" else "#4C72B0" for d in top["direccion"]]
    etiquetas = top["region_nombre"] + " [" + top["region"] + "]"
    ax.barh(etiquetas, top["RTE"], color=colores, alpha=0.85)
    for y, (_, f) in enumerate(top.iterrows()):
        lo = f["RTE_IC_inf"] if np.isfinite(f["RTE_IC_inf"]) else f["RTE"]
        hi = min(f["RTE_IC_sup"], f["RTE"] * 4) if np.isfinite(f["RTE_IC_sup"]) else f["RTE"]
        ax.plot([lo, hi], [y, y], color="black", lw=1.3, alpha=0.75)
    ax.axvline(1, color="black", ls="--", lw=1.8, label="RTE = 1 (exactamente lo esperado)")
    ax.set_xlabel("Razon de tasas estandarizada (observado / esperado), IC 95 % exacto")
    ax.set_title("Regiones con actividad sismica fuera de lo esperado")
    ax.legend()
    plt.tight_layout()
    plt.show()

    n_sube = int((ranking["direccion"] == "AUMENTO").sum())
    n_baja = int((ranking["direccion"] == "DESCENSO").sum())
    peor = ranking.iloc[0]
    conclusion(
        f"{n_sube} regiones muestran un AUMENTO significativo de la tasa y {n_baja} un DESCENSO "
        f"significativo, siempre respecto al comportamiento esperado tras descontar la deriva "
        f"global del catalogo.\n"
        f"El caso mas extremo es {peor['region_nombre']} [{peor['region']}]: observo "
        f"{int(peor['n_rec'])} sismos en la ventana reciente frente a {peor['esperado_rec']:.1f} "
        f"esperados, es decir, {peor['RTE']:.2f} veces lo esperado (IC 95 %: "
        f"[{peor['RTE_IC_inf']:.2f} , {peor['RTE_IC_sup']:.2f}], z cuasi-Poisson = "
        f"{peor['z_quasi']:+.2f}, q = {peor['q_quasi']:.2e}).\n"
        f"El intervalo de confianza es tan informativo como el p-valor: cuando NO contiene el 1, "
        f"el cambio es estadisticamente solido, y su amplitud indica con que precision se conoce "
        f"la magnitud del cambio. Las regiones con DESCENSO merecen la misma atencion analitica: "
        f"pueden reflejar una disminucion real de la sismicidad de fondo, pero tambien el final "
        f"de una secuencia de replicas que inflaba artificialmente su linea base historica."
    )
else:
    conclusion(
        "Ninguna region supera el umbral de FDR bajo el criterio cuasi-Poisson estandarizado: la "
        "actividad reciente es compatible, region por region, con el comportamiento historico "
        "corregido por la deriva del catalogo. Este es un resultado NEGATIVO INFORMATIVO, no un "
        "fracaso del analisis: cuantifica que las fluctuaciones recientes caen dentro de la "
        "variabilidad natural del proceso."
    )

In [ ]:
titulo("30. Mapa de anomalias")

mapa = res.copy()
mapa["lat_c"] = mapa["region"].str.split("|").str[0].astype(float) + PASO_MALLA / 2
mapa["lon_c"] = mapa["region"].str.split("|").str[1].astype(float) + PASO_MALLA / 2
mapa["log_rt"] = np.log2(mapa["RTE"].replace(0, np.nan))

fig, ax = plt.subplots(figsize=(14, 7))
ax.scatter(mapa["lon_c"], mapa["lat_c"], s=14, color="#BBBBBB", alpha=0.6,
           label="Region evaluada (sin anomalia)")
sig = mapa[mapa["signif_quasi"]]
sc = ax.scatter(sig["lon_c"], sig["lat_c"],
                s=np.clip(np.abs(sig["z_quasi"]) * 22, 40, 700),
                c=sig["log_rt"], cmap="coolwarm", vmin=-1.0, vmax=1.0,
                edgecolors="black", linewidths=0.7, alpha=0.9, zorder=5)
plt.colorbar(sc, ax=ax, label="log2(RTE)   > 0 aumento   < 0 descenso")
for _, f in sig.sort_values("z_quasi", key=abs, ascending=False).head(10).iterrows():
    ax.annotate(f["region_nombre"], (f["lon_c"], f["lat_c"]), fontsize=7.5,
                xytext=(5, 5), textcoords="offset points")
ax.set_xlim(-180, 180)
ax.set_ylim(-80, 85)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Distribucion geografica de las anomalias sismicas "
             "(tamano = |z| cuasi-Poisson, color = direccion del cambio)")
ax.grid(alpha=0.25)
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

conclusion(
    "El mapa permite distinguir entre una anomalia AISLADA (una celda suelta, mas probablemente "
    "una secuencia local de replicas) y una anomalia REGIONALMENTE COHERENTE (varias celdas "
    "contiguas del mismo color, que sugiere un cambio a escala de segmento tectonico y es un "
    "hallazgo mucho mas robusto). La coherencia espacial funciona como una validacion "
    "independiente que ningun p-valor por si solo puede aportar: celdas vecinas son unidades de "
    "analisis distintas, de modo que su concordancia no es un artefacto del mismo test."
)

## 6.4 Segunda vía de anomalía: cambio en el *b-value* (test de Utsu)

Una región puede tener **el mismo número de sismos** y aun así estar comportándose de forma
anómala: si el reparto entre sismos pequeños y grandes cambia, cambia el **riesgo**.



In [ ]:
titulo("31. Comparacion del b-value historico vs reciente (test de Utsu)")


def test_utsu(b1, n1, b2, n2):
    """Utsu (1992): contraste de igualdad de dos b-values basado en AIC."""
    if not all(np.isfinite([b1, b2])) or min(n1, n2) < 2 or b1 <= 0 or b2 <= 0:
        return np.nan, np.nan
    N = n1 + n2
    dA = (-2 * N * np.log(N)
          + 2 * n1 * np.log(n1 + n2 * b1 / b2)
          + 2 * n2 * np.log(n1 * b2 / b1 + n2)
          - 2)
    return dA, float(min(np.exp(-dA / 2 - 2), 1.0))


MIN_EVENTOS_B = 50
filas_b = []
for reg, g in dfm[dfm["region"].isin(regiones_validas)].groupby("region"):
    mh = g.loc[g["periodo"] == "HISTORICO", "mag"]
    mr = g.loc[g["periodo"] == "RECIENTE", "mag"]
    if len(mh) < MIN_EVENTOS_B or len(mr) < MIN_EVENTOS_B:
        continue
    b1, s1, n1 = b_value_aki(mh)
    b2, s2, n2 = b_value_aki(mr)
    if not np.isfinite(b1) or not np.isfinite(b2):
        continue
    dA, p_utsu = test_utsu(b1, n1, b2, n2)
    filas_b.append({"region": reg, "region_nombre": g["region_nombre"].iloc[0],
                    "n_hist": n1, "n_rec": n2,
                    "b_hist": b1, "sigma_b_hist": s1,
                    "b_rec": b2, "sigma_b_rec": s2,
                    "delta_b": b2 - b1, "dAIC": dA, "p_utsu": p_utsu})

bvals = pd.DataFrame(filas_b)
if len(bvals):
    bvals["q_utsu"] = bh_fdr(bvals["p_utsu"].fillna(1.0))
    bvals["signif_b"] = bvals["q_utsu"] < ALPHA
    bvals["interpretacion"] = np.where(
        ~bvals["signif_b"], "sin cambio",
        np.where(bvals["delta_b"] < 0,
                 "b BAJA: mas proporcion de sismos GRANDES",
                 "b SUBE: mas proporcion de sismos pequenos"))

    print(f"Regiones con b-value estimable en ambos periodos: {len(bvals)}")
    print(f"Con cambio significativo del b-value (FDR<{ALPHA}): {int(bvals['signif_b'].sum())}")
    display(bvals[bvals["signif_b"]].sort_values("delta_b")
            .head(15)[["region_nombre", "region", "n_hist", "n_rec", "b_hist", "b_rec",
                       "delta_b", "p_utsu", "q_utsu", "interpretacion"]].round(4))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
    axes[0].errorbar(bvals["b_hist"], bvals["b_rec"],
                     xerr=1.96 * bvals["sigma_b_hist"], yerr=1.96 * bvals["sigma_b_rec"],
                     fmt="o", ms=4, alpha=0.45, color="#4C72B0", ecolor="#BBBBBB", lw=0.8)
    sigb = bvals[bvals["signif_b"]]
    axes[0].scatter(sigb["b_hist"], sigb["b_rec"], s=60, color="#C44E52",
                    edgecolors="black", lw=0.6, zorder=5, label="cambio significativo")
    lim = [bvals[["b_hist", "b_rec"]].min().min() * 0.95,
           bvals[["b_hist", "b_rec"]].max().max() * 1.05]
    axes[0].plot(lim, lim, color="black", ls="--", lw=1.5, label="sin cambio (b_rec = b_hist)")
    axes[0].set_xlabel("b-value HISTORICO")
    axes[0].set_ylabel("b-value RECIENTE")
    axes[0].set_title("b-value por region: historico vs reciente (IC 95 % de Aki)")
    axes[0].legend(fontsize=8)

    sns.histplot(bvals["delta_b"], bins=28, color="#4C72B0", ax=axes[1])
    axes[1].axvline(0, color="black", ls="--", lw=1.6)
    axes[1].axvline(bvals["delta_b"].median(), color="#C44E52", lw=2,
                    label=f"mediana = {bvals['delta_b'].median():+.3f}")
    axes[1].set_title("Distribucion del cambio en el b-value (b_rec - b_hist)")
    axes[1].set_xlabel("delta b")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    n_baja_b = int(((bvals["delta_b"] < 0) & bvals["signif_b"]).sum())
    conclusion(
        f"De {len(bvals)} regiones con muestra suficiente (>= {MIN_EVENTOS_B} sismos en cada "
        f"periodo), {int(bvals['signif_b'].sum())} cambiaron significativamente su b-value y en "
        f"{n_baja_b} de ellas el b-value BAJO.\n"
        f"Un descenso del b-value es un tipo de anomalia CUALITATIVAMENTE DISTINTO al aumento de "
        f"la tasa: la region no produce necesariamente mas sismos, sino que el reparto de energia "
        f"se desplaza hacia eventos de mayor tamano. En la literatura sismologica esto se asocia "
        f"a un aumento del esfuerzo diferencial acumulado en la falla.\n"
        f"Por eso el diagnostico final NO puede basarse solo en el conteo: una region con tasa "
        f"normal pero con b-value en descenso tambien esta 'fuera de lo esperado', y en terminos "
        f"de riesgo puede ser mas preocupante que otra con muchos sismos pequenos."
    )
else:
    conclusion("Ninguna region alcanza el minimo de eventos requerido para estimar el b-value "
               "de forma fiable en ambos periodos. El criterio de tasa (seccion 6.3) es entonces "
               "el unico disponible.")

## 6.5 ¿Cuándo empezó la desviación? Carta de control de Shewhart



In [ ]:
titulo("32. Cartas de control de las regiones mas anomalas")

candidatas = (ranking.head(4) if len(ranking)
              else res.sort_values("z_quasi", key=abs, ascending=False).head(4))

fig, axes = plt.subplots(len(candidatas), 1, figsize=(12.5, 3.1 * len(candidatas)), sharex=True)
axes = np.atleast_1d(axes)

for ax, (_, fila) in zip(axes, candidatas.iterrows()):
    reg = fila["region"]
    g = panel[panel["region"] == reg].sort_values("mes")
    serie = g.set_index("fecha")["n_eventos"]
    h = g[g["periodo"] == "HISTORICO"]["n_eventos"]

    lam = h.mean()
    phi = max(float(res.loc[res["region"] == reg, "phi"].iloc[0]), 1.0)
    sigma = np.sqrt(phi * lam)
    LCS, LCI = lam + 3 * sigma, max(lam - 3 * sigma, 0)
    zona_a = lam + 2 * sigma                      # zona de aviso (2 sigma)

    ax.plot(serie.index, serie.values, lw=0.9, color="#555555")
    rec = g[g["periodo"] == "RECIENTE"]
    fuera = rec[rec["n_eventos"] > LCS]
    ax.scatter(fuera["fecha"], fuera["n_eventos"], s=45, color="#C44E52", zorder=6,
               label=f"fuera de control ({len(fuera)} meses)")
    ax.axhline(lam, color="#4C72B0", lw=1.6, label=f"linea central lambda = {lam:.2f}")
    ax.axhline(LCS, color="#C44E52", ls="--", lw=1.5,
               label=f"LCS = lambda + 3s = {LCS:.2f}  (s corregida por phi={phi:.2f})")
    ax.axhline(zona_a, color="#DD8452", ls=":", lw=1.2, label="zona de aviso (2s)")
    if LCI > 0:
        ax.axhline(LCI, color="#C44E52", ls="--", lw=1.5)
    ax.axvspan(MES_CORTE.to_timestamp(), serie.index.max(), color="#FFF2CC", alpha=0.45, zorder=0)
    ax.set_title(f"{fila['region_nombre']} [{reg}] — "
                 f"RTE = {fila['RTE']:.2f}, z = {fila['z_quasi']:+.2f}", fontsize=10)
    ax.set_ylabel("sismos/mes")
    ax.legend(fontsize=7, ncol=2, loc="upper left")

axes[-1].set_xlabel("Fecha  (franja amarilla = periodo RECIENTE, evaluado out-of-sample)")
plt.tight_layout()
plt.show()

# --- Reglas de Western Electric sobre la region mas anomala ------------------
reg0 = candidatas.iloc[0]["region"]
g0 = panel[panel["region"] == reg0].sort_values("mes")
h0 = g0[g0["periodo"] == "HISTORICO"]["n_eventos"]
lam0 = h0.mean()
phi0 = max(float(res.loc[res["region"] == reg0, "phi"].iloc[0]), 1.0)
s0 = np.sqrt(phi0 * lam0)
r0 = g0[g0["periodo"] == "RECIENTE"].copy()
r0["z"] = (r0["n_eventos"] - lam0) / s0

regla1 = int((r0["z"].abs() > 3).sum())
regla2 = int(((r0["z"] > 2).rolling(3).sum() >= 2).sum())
regla3 = int(((r0["z"] > 0).rolling(8).sum() == 8).sum())

print(f"Region analizada: {candidatas.iloc[0]['region_nombre']} [{reg0}]")
print(f"  Regla 1 (1 punto fuera de 3 sigma)          : {regla1} meses")
print(f"  Regla 2 (2 de 3 puntos consecutivos > 2 s)  : {regla2} ocurrencias")
print(f"  Regla 3 (8 puntos consecutivos sobre lambda): {regla3} ocurrencias")

conclusion(
    f"La carta de control anade la dimension TEMPORAL que el test agregado no ofrece: permite "
    f"distinguir un pico aislado (un unico mes fuera de limites, tipicamente un sismo principal "
    f"con sus replicas) de una DESVIACION SOSTENIDA (varios meses consecutivos por encima de la "
    f"linea central), que es la senal realmente preocupante.\n"
    f"En la region mas anomala se activan la regla 1 en {regla1} meses, la regla 2 en {regla2} "
    f"ocasiones y la regla 3 en {regla3}. "
    + ("La presencia de rachas (reglas 2 y 3) indica un cambio de NIVEL del proceso y no una "
       "fluctuacion puntual: es la evidencia mas fuerte de que la region opera en un regimen "
       "distinto al historico."
       if (regla2 + regla3) > 0 else
       "Sin rachas activadas, la desviacion se concentra en meses puntuales, compatible con "
       "secuencias de replicas aisladas mas que con un cambio permanente de regimen.")
    + f"\nNotese que los limites se calcularon SOLO con el historico y se aplican al periodo "
      f"reciente sin reajustarlos: es una validacion out-of-sample genuina."
)

## 6.6 Diagnóstico combinado: construcción del *target* categórico `es_anomala`


In [ ]:
titulo("33. Clasificacion final de las regiones")

diag = res[["region", "region_nombre", "n_hist", "n_rec", "esperado_rec", "RTE",
            "z_quasi", "q_quasi", "signif_quasi", "phi"]].copy()

if len(bvals):
    diag = diag.merge(bvals[["region", "b_hist", "b_rec", "delta_b", "q_utsu", "signif_b"]],
                      on="region", how="left")
else:
    diag[["b_hist", "b_rec", "delta_b", "q_utsu"]] = np.nan
    diag["signif_b"] = False
diag["signif_b"] = diag["signif_b"].fillna(False)


def clasificar(f):
    tasa_alta = f["signif_quasi"] and f["RTE"] > 1
    tasa_baja = f["signif_quasi"] and f["RTE"] < 1
    b_baja = bool(f["signif_b"]) and (f["delta_b"] < 0)
    b_sube = bool(f["signif_b"]) and (f["delta_b"] > 0)
    if tasa_alta and b_baja:
        return "1. CRITICA: mas sismos Y mas grandes"
    if tasa_alta:
        return "2. ACTIVACION: tasa por encima de lo esperado"
    if b_baja:
        return "3. VIGILANCIA: b-value en descenso (sismos mas grandes)"
    if tasa_baja:
        return "4. QUIESCENCIA: tasa por debajo de lo esperado"
    if b_sube:
        return "5. b-value en ascenso (sismos mas pequenos)"
    return "6. NORMAL: dentro de lo esperado"


diag["diagnostico"] = diag.apply(clasificar, axis=1)
diag["es_anomala"] = (~diag["diagnostico"].str.startswith("6")).astype(int)

resumen_diag = (diag["diagnostico"].value_counts().sort_index()
                .to_frame("regiones"))
resumen_diag["%"] = (resumen_diag["regiones"] / len(diag) * 100).round(1)
display(resumen_diag)

fig, ax = plt.subplots(figsize=(10, 4))
colores_diag = {"1.": "#8B0000", "2.": "#C44E52", "3.": "#DD8452",
                "4.": "#4C72B0", "5.": "#64B5CD", "6.": "#CCCCCC"}
ax.barh(resumen_diag.index.astype(str), resumen_diag["regiones"],
        color=[colores_diag[i[:2]] for i in resumen_diag.index])
for y, v in enumerate(resumen_diag["regiones"]):
    ax.text(v + max(resumen_diag["regiones"]) * 0.01, y, f"{v}  ({v / len(diag) * 100:.1f} %)",
            va="center", fontsize=9)
ax.set_xlabel("N.o de regiones")
ax.set_title("Clasificacion final de las regiones segun su comportamiento reciente")
ax.set_xlim(0, max(resumen_diag["regiones"]) * 1.25)
plt.tight_layout()
plt.show()

n_anom = int(diag["es_anomala"].sum())
n_crit = int(diag["diagnostico"].str.startswith("1").sum())
conclusion(
    f"El target categorico 'es_anomala' queda construido sobre criterios estadisticos explicitos "
    f"y auditables, no sobre un umbral arbitrario: {n_anom} de {len(diag)} regiones "
    f"({n_anom / len(diag) * 100:.1f} %) estan fuera de lo esperado.\n"
    f"La categoria CRITICA ({n_crit} regiones) es la que concentra el riesgo: son regiones donde "
    f"coinciden DOS senales estadisticamente independientes entre si (mas eventos de los "
    f"esperados Y un desplazamiento del reparto de magnitudes hacia sismos grandes). La "
    f"concurrencia de dos contrastes independientes es una evidencia mucho mas robusta que "
    f"cualquiera de los dos por separado, porque la probabilidad conjunta de un falso positivo "
    f"doble es el producto de las individuales.\n"
    f"La categoria QUIESCENCIA no debe descartarse: en sismologia, un periodo anomalo de calma "
    f"(quiescencia sismica) ha sido documentado como precursor en algunas secuencias, aunque su "
    f"valor predictivo sigue siendo objeto de debate cientifico."
)

---
# 7. Fase 6 — Transformaciones



In [ ]:
titulo("34. Transformaciones estabilizadoras de varianza")

serie_reg = (panel.pivot_table(index="mes", columns="region", values="n_eventos",
                               aggfunc="sum").fillna(0))
N = panel_hist["n_eventos"].astype(float)

transformaciones = {
    "Original N": N,
    "log(1+N)": np.log1p(N),
    "Anscombe 2*sqrt(N+3/8)": 2 * np.sqrt(N + 3 / 8),
    "Freeman-Tukey sqrt(N)+sqrt(N+1)": np.sqrt(N) + np.sqrt(N + 1),
}
lam_bc = np.nan
try:
    bc, lam_bc = stats.boxcox(N + 1)
    transformaciones[f"Box-Cox (lambda={lam_bc:.3f})"] = pd.Series(bc, index=N.index)
except Exception as e:
    print("Box-Cox no aplicable:", e)

# --- Criterio: exponente de la LEY DE POTENCIAS DE TAYLOR ---------------------
# Se regresa log(varianza de la variable TRANSFORMADA en la region) sobre
# log(nivel de actividad ORIGINAL de la region, lambda_r). Anclar el eje x al nivel
# original es lo que hace comparables entre si a las distintas transformaciones.
#    beta = 1  -> varianza proporcional al nivel (Poisson puro: heterocedastico)
#    beta = 0  -> varianza CONSTANTE: homocedasticidad lograda
nivel_region = panel_hist.groupby("region")["n_eventos"].mean()   # lambda_r original

filas_t = []
for nombre, serie in transformaciones.items():
    tmp = panel_hist[["region"]].copy()
    tmp["v"] = np.asarray(serie, dtype=float)
    est = tmp.groupby("region")["v"].var().to_frame("var")
    est["nivel"] = nivel_region
    est = est[(est["var"] > 0) & (est["nivel"] > 0)].dropna()
    beta = (np.polyfit(np.log(est["nivel"]), np.log(est["var"]), 1)[0]
            if len(est) > 5 else np.nan)
    rho, _ = stats.spearmanr(est["nivel"], est["var"])
    # Dispersion relativa de las varianzas entre regiones: cuanto menor, mas homogeneas
    cv_var = est["var"].std() / est["var"].mean()
    grupos = [g["v"].values for _, g in tmp.groupby("region") if len(g) > 20]
    W, p_lev = stats.levene(*grupos[:60], center="median")
    s = pd.Series(serie).dropna()
    filas_t.append({
        "Transformacion": nombre,
        "beta de Taylor": beta,
        "CV de las varianzas": cv_var,
        "Levene W": W,
        "rho(nivel,var)": rho,
        "Asimetria": stats.skew(s),
        "Curtosis": stats.kurtosis(s),
        "Shapiro p (n=45, mediana)": float(np.median(
            [stats.shapiro(s.sample(45, random_state=i)).pvalue for i in range(60)])),
    })

tabla_trans = pd.DataFrame(filas_t).round(4)
display(tabla_trans)

fig, axes = plt.subplots(1, len(transformaciones), figsize=(3.4 * len(transformaciones), 3.5))
for ax, (nombre, serie) in zip(np.atleast_1d(axes), transformaciones.items()):
    tmp = panel_hist[["region"]].copy()
    tmp["v"] = np.asarray(serie, dtype=float)
    est = tmp.groupby("region")["v"].var().to_frame("var")
    est["nivel"] = nivel_region
    est = est[(est["var"] > 0) & (est["nivel"] > 0)].dropna()
    ax.scatter(est["nivel"], est["var"], s=13, alpha=0.6, color="#4C72B0")
    b_i = float(tabla_trans.loc[tabla_trans["Transformacion"] == nombre,
                                "beta de Taylor"].iloc[0])
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"{nombre}\nbeta = {b_i:+.2f}", fontsize=8.5)
    ax.set_xlabel("nivel de actividad lambda_r")
    ax.set_ylabel("varianza transformada")
plt.suptitle("Varianza frente al nivel de actividad (escala log-log): una nube PLANA "
             "(beta ~ 0) indica varianza estabilizada", y=1.04, fontweight="bold")
plt.tight_layout()
plt.show()

# Se excluye la variable original (es el punto de partida, no una opcion) y se descarta
# Box-Cox cuando su lambda cae fuera del rango interpretable: con exceso de ceros el
# ajuste es degenerado y la transformacion resultante no tiene lectura practica.
candidatas_t = tabla_trans[tabla_trans["Transformacion"] != "Original N"].copy()
box_cox_valida = np.isfinite(lam_bc) and (-0.25 <= lam_bc <= 1.25)
if not box_cox_valida:
    candidatas_t = candidatas_t[~candidatas_t["Transformacion"].str.startswith("Box-Cox")]
mejor_t = candidatas_t.reindex(
    candidatas_t["beta de Taylor"].abs().sort_values().index).iloc[0]
beta0 = float(tabla_trans.iloc[0]["beta de Taylor"])
conclusion(
    f"La variable original tiene un exponente de Taylor beta = {beta0:.3f}, muy proximo a 1: la "
    f"varianza crece PROPORCIONALMENTE a la media. Esa es exactamente la heterocedasticidad que "
    f"predice el modelo de Poisson (Var = E) y la razon por la que no puede usarse una desviacion "
    f"estandar comun para fijar limites de control validos en regiones con niveles de actividad "
    f"muy distintos.\n"
    f"La transformacion recomendada es '{mejor_t['Transformacion']}', que lleva el exponente a "
    f"beta = {mejor_t['beta de Taylor']:+.3f} (beta = 0 significa varianza constante). "
    + ((f"Box-Cox estima por maxima verosimilitud lambda = {lam_bc:.3f}. "
        + ("Al estar proximo a 0.5, confirma que la RAIZ CUADRADA es la potencia adecuada, que es "
           "justamente el fundamento teorico de la transformacion de Anscombe para datos de "
           "Poisson."
           if 0.25 <= lam_bc <= 0.75 else
           "Al estar proximo a 0, equivale a la transformacion LOGARITMICA."
           if -0.25 <= lam_bc < 0.25 else
           f"El valor se aleja del rango habitual (0 = log, 0.5 = raiz), lo que era previsible: "
           f"Box-Cox exige datos estrictamente positivos y aqui se aplico sobre N+1 con un "
           f"{(N == 0).mean() * 100:.0f} % de ceros. Con exceso de ceros, Box-Cox pierde sentido y "
           f"los estabilizadores especificos para conteos (Anscombe, Freeman-Tukey), derivados "
           f"analiticamente de la distribucion de Poisson, son preferibles."))
       if np.isfinite(lam_bc) else "")
    + f"\nADVERTENCIA IMPORTANTE: la transformacion mejora la homocedasticidad y la simetria, "
      f"PERO NO CONVIERTE LOS DATOS EN NORMALES (Shapiro sigue rechazando) y, sobre todo, "
      f"DISTORSIONA LA INTERPRETACION: 'raiz del numero de sismos' no significa nada para quien "
      f"toma decisiones. Por eso en este trabajo las transformaciones se usan como DIAGNOSTICO y "
      f"como base de los limites de control, mientras que los contrastes principales se hacen con "
      f"el modelo de Poisson sobre la escala ORIGINAL de conteos, que es exacto y directamente "
      f"interpretable. Transformar no es obligatorio cuando existe un modelo correcto para el "
      f"dato tal como es."
)

## 7.1 Retornos logarítmicos: la actividad sísmica como serie de "volatilidad"



In [ ]:
titulo("35. Retornos logaritmicos y volatilidad de la actividad sismica")

serie_glob = panel.groupby("mes")["n_eventos"].sum()
r = np.log1p(serie_glob).diff().dropna()
fechas_r = r.index.to_timestamp()
vol = r.rolling(12).std()

fig, axes = plt.subplots(3, 1, figsize=(12, 9))
axes[0].plot(serie_glob.index.to_timestamp(), serie_glob.values, lw=1.0, color="#4C72B0")
axes[0].axvline(MES_CORTE.to_timestamp(), color="black", ls="--", lw=1.4)
axes[0].set_title("Nivel: sismos por mes (serie global)")
axes[0].set_ylabel("N.o de sismos")

axes[1].plot(fechas_r, r.values, lw=0.9, color="#55A868")
axes[1].axhline(0, color="black", lw=0.9)
for k, c in [(2, "#DD8452"), (3, "#C44E52")]:
    axes[1].axhline(k * r.std(), color=c, ls="--", lw=1.2, label=f"+-{k} sigma")
    axes[1].axhline(-k * r.std(), color=c, ls="--", lw=1.2)
axes[1].set_title("Retornos logaritmicos r_t = ln(N_t + 1) - ln(N_t-1 + 1)")
axes[1].set_ylabel("retorno")
axes[1].legend(fontsize=8, ncol=2)

axes[2].plot(vol.index.to_timestamp(), vol.values, lw=1.6, color="#C44E52")
axes[2].axhline(vol.mean(), color="black", ls="--", lw=1.2,
                label=f"volatilidad media = {vol.mean():.3f}")
axes[2].axvline(MES_CORTE.to_timestamp(), color="black", ls=":", lw=1.4)
axes[2].set_title("Volatilidad movil de 12 meses (desviacion estandar de los retornos)")
axes[2].set_ylabel("volatilidad")
axes[2].set_xlabel("Fecha")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

vol_h = r[r.index < MES_CORTE].std()
vol_r = r[r.index >= MES_CORTE].std()
F_var = (vol_r ** 2) / (vol_h ** 2)
n1 = (r.index < MES_CORTE).sum() - 1
n2 = (r.index >= MES_CORTE).sum() - 1
p_F = 2 * min(stats.f.cdf(F_var, n2, n1), 1 - stats.f.cdf(F_var, n2, n1))
ks_r, p_ks_r = stats.kstest((r - r.mean()) / r.std(), "norm")

print(f"Volatilidad HISTORICA : {vol_h:.4f}")
print(f"Volatilidad RECIENTE  : {vol_r:.4f}   (razon = {vol_r / vol_h:.3f})")
print(f"Test F de igualdad de varianzas: F = {F_var:.3f} | "
      f"{veredicto(p_F, h0='misma volatilidad', h1='la volatilidad cambio')}")
print(f"Normalidad de los retornos (K-S): {veredicto(p_ks_r, h0='retornos normales', h1='no normales')}")
print(f"Asimetria de los retornos = {stats.skew(r):+.3f} | "
      f"Curtosis = {stats.kurtosis(r):+.3f}")

conclusion(
    f"Los retornos oscilan en torno a cero (no hay tendencia sistematica en el cambio relativo) "
    f"pero exhiben CURTOSIS = {stats.kurtosis(r):+.2f}. "
    + ("Un exceso de curtosis positivo significa colas pesadas: los meses de cambio brusco son "
       "mucho mas frecuentes de lo que predice una normal, exactamente el mismo fenomeno de "
       "'colas gordas' que se observa en series financieras."
       if stats.kurtosis(r) > 0.5 else
       "La curtosis es moderada, de modo que los cambios relativos extremos no son "
       "desproporcionadamente frecuentes.")
    + f"\nLa volatilidad reciente es {vol_r / vol_h:.2f} veces la historica y el test F "
    + ("CONFIRMA que el cambio es significativo: la actividad no solo pudo cambiar de nivel, "
       "cambio tambien su REGULARIDAD, lo que constituye una tercera dimension de anomalia."
       if p_F < ALPHA else
       "NO detecta un cambio significativo: la variabilidad relativa mes a mes se mantiene "
       "estable, de modo que las anomalias detectadas son de NIVEL y no de regularidad.")
    + f"\nAdemas se observa AGRUPAMIENTO DE VOLATILIDAD (periodos turbulentos seguidos de "
      f"periodos tranquilos), el analogo sismico del clustering de volatilidad financiera; es la "
      f"misma senal de dependencia temporal ya detectada en la autocorrelacion y en la "
      f"sobredispersion, vista ahora en la escala de los cambios relativos."
)